In [1]:
# Cell 1. Setup and sleep episode index for strict pre-sleep forecasting
# 원하는 결과:
# - Design C 입력 소스 파일 존재 여부를 확인한다.
# - sleep episode 단위 index를 만든다.
# - prediction cutoff가 sleep_start_datetime임을 명시한다.
# - calendar_date가 sleep end date임을 다시 검증한다.
# - 아직 파일 저장은 하지 않는다.

from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

SLEEP_TARGET_PATH = PROCESSED_DIR / "daily_aggregates" / "sleep_daily_target.csv"
MODELING_DAILY_PATH = PROCESSED_DIR / "modeling_dataset_daily.csv"
DESIGN_SPEC_PATH = PROCESSED_DIR / "pre_sleep_forecasting" / "pre_sleep_design_c_feature_spec.csv"

OUTPUT_DIR = PROCESSED_DIR / "pre_sleep_forecasting" / "design_c_dataset"

ID_COL = "participant_object_id"
CALENDAR_DATE_COL = "calendar_date"
START_TIME_COL = "startTime"
END_TIME_COL = "endTime"
TARGET_COL = "good_sleep_label"

input_paths = {
    "sleep_target": SLEEP_TARGET_PATH,
    "modeling_daily": MODELING_DAILY_PATH,
    "design_feature_spec": DESIGN_SPEC_PATH,
}

print("=== Input Path Check ===")
for name, path in input_paths.items():
    print(f"{name}: {path.relative_to(PROJECT_ROOT)} | exists={path.exists()}")

sleep_df = pd.read_csv(SLEEP_TARGET_PATH, encoding="utf-8-sig")
daily_df = pd.read_csv(MODELING_DAILY_PATH, encoding="utf-8-sig")
feature_spec_df = pd.read_csv(DESIGN_SPEC_PATH, encoding="utf-8-sig")

sleep_df[CALENDAR_DATE_COL] = pd.to_datetime(sleep_df[CALENDAR_DATE_COL])
sleep_df[START_TIME_COL] = pd.to_datetime(sleep_df[START_TIME_COL])
sleep_df[END_TIME_COL] = pd.to_datetime(sleep_df[END_TIME_COL])

sleep_df = sleep_df.sort_values([ID_COL, START_TIME_COL, END_TIME_COL]).reset_index(drop=True)

sleep_episode_df = sleep_df[
    [
        ID_COL,
        CALENDAR_DATE_COL,
        START_TIME_COL,
        END_TIME_COL,
        TARGET_COL,
    ]
].copy()

sleep_episode_df = sleep_episode_df.rename(
    columns={
        START_TIME_COL: "sleep_start_datetime",
        END_TIME_COL: "sleep_end_datetime",
    }
)

sleep_episode_df["sleep_start_date"] = sleep_episode_df["sleep_start_datetime"].dt.normalize()
sleep_episode_df["sleep_end_date"] = sleep_episode_df["sleep_end_datetime"].dt.normalize()
sleep_episode_df["prediction_cutoff_datetime"] = sleep_episode_df["sleep_start_datetime"]

sleep_episode_df["sleep_episode_id"] = (
    sleep_episode_df[ID_COL].astype(str)
    + "__"
    + sleep_episode_df["sleep_start_datetime"].dt.strftime("%Y%m%d%H%M%S")
    + "__"
    + sleep_episode_df["sleep_end_datetime"].dt.strftime("%Y%m%d%H%M%S")
)

sleep_episode_df["cross_midnight"] = (
    sleep_episode_df["sleep_start_date"] != sleep_episode_df["sleep_end_date"]
)

sleep_episode_df["calendar_matches_sleep_start_date"] = (
    sleep_episode_df[CALENDAR_DATE_COL].dt.normalize()
    == sleep_episode_df["sleep_start_date"]
)

sleep_episode_df["calendar_matches_sleep_end_date"] = (
    sleep_episode_df[CALENDAR_DATE_COL].dt.normalize()
    == sleep_episode_df["sleep_end_date"]
)

episode_summary = pd.DataFrame(
    [
        {"metric": "episodes", "value": len(sleep_episode_df)},
        {"metric": "participants", "value": sleep_episode_df[ID_COL].nunique()},
        {"metric": "unique_sleep_episode_ids", "value": sleep_episode_df["sleep_episode_id"].nunique()},
        {"metric": "target_missing_rows", "value": sleep_episode_df[TARGET_COL].isna().sum()},
        {"metric": "sleep_start_missing_rows", "value": sleep_episode_df["sleep_start_datetime"].isna().sum()},
        {"metric": "sleep_end_missing_rows", "value": sleep_episode_df["sleep_end_datetime"].isna().sum()},
        {"metric": "cross_midnight_rows", "value": sleep_episode_df["cross_midnight"].sum()},
        {"metric": "cross_midnight_ratio", "value": sleep_episode_df["cross_midnight"].mean()},
        {
            "metric": "calendar_matches_sleep_start_date_rows",
            "value": sleep_episode_df["calendar_matches_sleep_start_date"].sum(),
        },
        {
            "metric": "calendar_matches_sleep_end_date_rows",
            "value": sleep_episode_df["calendar_matches_sleep_end_date"].sum(),
        },
    ]
)

target_distribution = (
    sleep_episode_df[TARGET_COL]
    .value_counts(dropna=False)
    .rename_axis(TARGET_COL)
    .reset_index(name="rows")
)
target_distribution["ratio"] = target_distribution["rows"] / len(sleep_episode_df)

print("\n=== Loaded Shapes ===")
print("sleep_df:", sleep_df.shape)
print("daily_df:", daily_df.shape)
print("feature_spec_df:", feature_spec_df.shape)

print("\n=== Episode Summary ===")
display(episode_summary)

print("\n=== Target Distribution ===")
display(target_distribution)

print("\n=== Episode Preview ===")
display(
    sleep_episode_df[
        [
            "sleep_episode_id",
            ID_COL,
            "sleep_start_datetime",
            "sleep_end_datetime",
            "sleep_start_date",
            "sleep_end_date",
            "prediction_cutoff_datetime",
            TARGET_COL,
            "cross_midnight",
        ]
    ].head(10)
)

print("\n=== Design C Included Feature Groups ===")
display(
    feature_spec_df.loc[
        feature_spec_df["include_in_first_dataset"].astype(bool),
        ["feature_group", "source", "time_window", "leakage_risk", "notes"],
    ].reset_index(drop=True)
)

duplicate_episode_count = len(sleep_episode_df) - sleep_episode_df["sleep_episode_id"].nunique()
problem_rows = sleep_episode_df[
    sleep_episode_df["sleep_start_datetime"].isna()
    | sleep_episode_df["sleep_end_datetime"].isna()
    | sleep_episode_df[TARGET_COL].isna()
    | sleep_episode_df["sleep_episode_id"].duplicated(keep=False)
]

print("\n=== Validation Checks ===")
print("duplicate sleep_episode_id rows:", duplicate_episode_count)
print("problem rows:", len(problem_rows))

if len(problem_rows) > 0:
    display(problem_rows.head(20))
else:
    print("basic episode index validation passed")

=== Input Path Check ===
sleep_target: data\processed\daily_aggregates\sleep_daily_target.csv | exists=True
modeling_daily: data\processed\modeling_dataset_daily.csv | exists=True
design_feature_spec: data\processed\pre_sleep_forecasting\pre_sleep_design_c_feature_spec.csv | exists=True

=== Loaded Shapes ===
sleep_df: (3551, 28)
daily_df: (3551, 130)
feature_spec_df: (8, 7)

=== Episode Summary ===


,metric,value
0,episodes,3551.000000
1,participants,69.000000
2,unique_sleep_episode_ids,3551.000000
3,target_missing_rows,0.000000
4,sleep_start_missing_rows,0.000000
5,sleep_end_missing_rows,0.000000
6,cross_midnight_rows,1349.000000
7,cross_midnight_ratio,0.379893
8,calendar_matches_sleep_start_date_rows,2202.000000
9,calendar_matches_sleep_end_date_rows,3551.000000



=== Target Distribution ===


,good_sleep_label,rows,ratio
0,0,2153,0.606308
1,1,1398,0.393692



=== Episode Preview ===


,sleep_episode_id,participant_object_id,sleep_start_datetime,sleep_end_datetime,sleep_start_date,sleep_end_date,prediction_cutoff_datetime,good_sleep_label,cross_midnight
0,621e2e8e67b776a24055b564__20210524004000__2021...,621e2e8e67b776a24055b564,2021-05-24 00:40:00,2021-05-24 09:21:00,2021-05-24,2021-05-24,2021-05-24 00:40:00,1,False
1,621e2e8e67b776a24055b564__20210524234830__2021...,621e2e8e67b776a24055b564,2021-05-24 23:48:30,2021-05-25 08:56:30,2021-05-24,2021-05-25,2021-05-24 23:48:30,1,True
2,621e2e8e67b776a24055b564__20210525234630__2021...,621e2e8e67b776a24055b564,2021-05-25 23:46:30,2021-05-26 09:06:30,2021-05-25,2021-05-26,2021-05-25 23:46:30,1,True
3,621e2e8e67b776a24055b564__20210526232130__2021...,621e2e8e67b776a24055b564,2021-05-26 23:21:30,2021-05-27 09:48:30,2021-05-26,2021-05-27,2021-05-26 23:21:30,1,True
4,621e2e8e67b776a24055b564__20210527233700__2021...,621e2e8e67b776a24055b564,2021-05-27 23:37:00,2021-05-28 08:58:30,2021-05-27,2021-05-28,2021-05-27 23:37:00,1,True
5,621e2e8e67b776a24055b564__20210528235000__2021...,621e2e8e67b776a24055b564,2021-05-28 23:50:00,2021-05-29 08:48:00,2021-05-28,2021-05-29,2021-05-28 23:50:00,1,True
6,621e2e8e67b776a24055b564__20210530003000__2021...,621e2e8e67b776a24055b564,2021-05-30 00:30:00,2021-05-30 09:45:30,2021-05-30,2021-05-30,2021-05-30 00:30:00,1,False
7,621e2e8e67b776a24055b564__20210530232930__2021...,621e2e8e67b776a24055b564,2021-05-30 23:29:30,2021-05-31 09:08:30,2021-05-30,2021-05-31,2021-05-30 23:29:30,1,True
8,621e2e8e67b776a24055b564__20210531234230__2021...,621e2e8e67b776a24055b564,2021-05-31 23:42:30,2021-06-01 09:26:30,2021-05-31,2021-06-01,2021-05-31 23:42:30,1,True
9,621e2e8e67b776a24055b564__20210601235830__2021...,621e2e8e67b776a24055b564,2021-06-01 23:58:30,2021-06-02 09:32:30,2021-06-01,2021-06-02,2021-06-01 23:58:30,1,True



=== Design C Included Feature Groups ===


,feature_group,source,time_window,leakage_risk,notes
0,pre_sleep_steps,MongoDB fitbit type=steps,sleep_start_date 00:00 <= timestamp < sleep_st...,low,Strictly before sleep onset.
1,pre_sleep_calories,MongoDB fitbit type=calories,sleep_start_date 00:00 <= timestamp < sleep_st...,low,Strictly before sleep onset.
2,pre_sleep_heart_rate,MongoDB fitbit type=heart_rate,sleep_start_date 00:00 <= timestamp < sleep_st...,low,Strictly before sleep onset. Confidence can be...
3,pre_sleep_window_timing,sleep_daily_target.csv startTime,known at prediction time,low,Calendar/time features are known before predic...
4,previous_day_daily_activity,processed daily aggregates,sleep_start_date - 1 day,low,Already available before sleep onset.
5,rolling_history,Design C pre-sleep features and previous-day d...,previous 3/7 available days before current sle...,low,Must not include the target sleep episode outc...



=== Validation Checks ===
duplicate sleep_episode_id rows: 0
problem rows: 0
basic episode index validation passed


In [2]:
# Cell 2. Previous-day daily feature join for pre-sleep forecasting
# 원하는 결과:
# - sleep_start_date - 1 day 기준으로 daily aggregate feature를 붙인다.
# - 첫 Design C dataset에 넣을 previous-day daily activity/resting-HR feature 후보를 확정한다.
# - join coverage, missingness, target distribution을 확인한다.
# - 아직 파일 저장은 하지 않는다.

from pandas.api.types import is_numeric_dtype

DAILY_DATE_COL = "calendar_date"

daily_df = daily_df.copy()
daily_df[DAILY_DATE_COL] = pd.to_datetime(daily_df[DAILY_DATE_COL]).dt.normalize()

episode_daily_df = sleep_episode_df.copy()
episode_daily_df["previous_day_feature_date"] = (
    episode_daily_df["sleep_start_date"] - pd.Timedelta(days=1)
)

previous_day_candidate_features = [
    "resting_hr_resting_heart_rate_mean",
    "resting_hr_error_mean",
    "resting_hr_record_count",
    "lightly_active_minutes_sum",
    "moderately_active_minutes_sum",
    "sedentary_minutes_sum",
    "very_active_minutes_sum",
    "steps_sum",
    "steps_record_count",
    "calories_sum",
    "calories_record_count",
]

available_previous_day_features = [
    col for col in previous_day_candidate_features
    if col in daily_df.columns
]

missing_previous_day_features = [
    col for col in previous_day_candidate_features
    if col not in daily_df.columns
]

daily_join_cols = [ID_COL, DAILY_DATE_COL] + available_previous_day_features

previous_day_daily_features = daily_df[daily_join_cols].copy()
previous_day_daily_features = previous_day_daily_features.rename(
    columns={
        DAILY_DATE_COL: "previous_day_feature_date",
        **{
            col: f"previous_day_{col}"
            for col in available_previous_day_features
        },
    }
)

pre_sleep_design_df = episode_daily_df.merge(
    previous_day_daily_features,
    on=[ID_COL, "previous_day_feature_date"],
    how="left",
    indicator="previous_day_daily_join_status",
)

previous_day_feature_cols = [
    f"previous_day_{col}"
    for col in available_previous_day_features
]

pre_sleep_design_df["previous_day_daily_joined"] = (
    pre_sleep_design_df["previous_day_daily_join_status"] == "both"
)

join_summary = pd.DataFrame(
    [
        {"metric": "episodes", "value": len(pre_sleep_design_df)},
        {
            "metric": "previous_day_daily_joined_rows",
            "value": int(pre_sleep_design_df["previous_day_daily_joined"].sum()),
        },
        {
            "metric": "previous_day_daily_join_coverage",
            "value": float(pre_sleep_design_df["previous_day_daily_joined"].mean()),
        },
        {
            "metric": "available_previous_day_features",
            "value": len(available_previous_day_features),
        },
        {
            "metric": "missing_previous_day_features",
            "value": len(missing_previous_day_features),
        },
    ]
)

previous_day_missing_summary = (
    pre_sleep_design_df[previous_day_feature_cols]
    .isna()
    .mean()
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "feature"})
    .sort_values(["missing_rate", "feature"], ascending=[False, True])
)

previous_day_numeric_check = pd.DataFrame(
    [
        {
            "feature": col,
            "is_numeric": is_numeric_dtype(pre_sleep_design_df[col]),
            "non_missing_rows": int(pre_sleep_design_df[col].notna().sum()),
        }
        for col in previous_day_feature_cols
    ]
)

join_target_summary = (
    pre_sleep_design_df
    .groupby("previous_day_daily_joined")
    .agg(
        rows=(TARGET_COL, "size"),
        participants=(ID_COL, "nunique"),
        target_mean=(TARGET_COL, "mean"),
        target_0=(TARGET_COL, lambda x: int((x == 0).sum())),
        target_1=(TARGET_COL, lambda x: int((x == 1).sum())),
    )
    .reset_index()
)

print("=== Previous-Day Daily Feature Join Summary ===")
display(join_summary)

print("\n=== Available Previous-Day Candidate Features ===")
print(available_previous_day_features)

print("\n=== Missing Previous-Day Candidate Features ===")
print(missing_previous_day_features)

print("\n=== Previous-Day Feature Missing Rate ===")
display(previous_day_missing_summary)

print("\n=== Previous-Day Feature Numeric Check ===")
display(previous_day_numeric_check)

print("\n=== Join Coverage And Target Distribution ===")
display(join_target_summary)

print("\n=== Preview Joined Dataset Columns ===")
display(
    pre_sleep_design_df[
        [
            "sleep_episode_id",
            ID_COL,
            "sleep_start_datetime",
            "previous_day_feature_date",
            "previous_day_daily_joined",
            TARGET_COL,
        ]
        + previous_day_feature_cols
    ].head(10)
)

problem_previous_day_features = previous_day_numeric_check[
    ~previous_day_numeric_check["is_numeric"]
]

print("\n=== Validation Checks ===")
print("non-numeric previous-day features:", len(problem_previous_day_features))
print("previous-day feature columns:", len(previous_day_feature_cols))

if len(problem_previous_day_features) > 0:
    display(problem_previous_day_features)
else:
    print("previous-day daily feature validation passed")

=== Previous-Day Daily Feature Join Summary ===


,metric,value
0,episodes,3551.000000
1,previous_day_daily_joined_rows,3101.000000
2,previous_day_daily_join_coverage,0.873275
3,available_previous_day_features,11.000000
4,missing_previous_day_features,0.000000



=== Available Previous-Day Candidate Features ===
['resting_hr_resting_heart_rate_mean', 'resting_hr_error_mean', 'resting_hr_record_count', 'lightly_active_minutes_sum', 'moderately_active_minutes_sum', 'sedentary_minutes_sum', 'very_active_minutes_sum', 'steps_sum', 'steps_record_count', 'calories_sum', 'calories_record_count']

=== Missing Previous-Day Candidate Features ===
[]

=== Previous-Day Feature Missing Rate ===


,feature,missing_rate
8,previous_day_steps_record_count,0.128978
7,previous_day_steps_sum,0.128978
10,previous_day_calories_record_count,0.126725
9,previous_day_calories_sum,0.126725
3,previous_day_lightly_active_minutes_sum,0.126725
4,previous_day_moderately_active_minutes_sum,0.126725
1,previous_day_resting_hr_error_mean,0.126725
2,previous_day_resting_hr_record_count,0.126725
0,previous_day_resting_hr_resting_heart_rate_mean,0.126725
5,previous_day_sedentary_minutes_sum,0.126725



=== Previous-Day Feature Numeric Check ===


...


=== Join Coverage And Target Distribution ===


,previous_day_daily_joined,rows,participants,target_mean,target_0,target_1
0,False,450,69,0.353333,291,159
1,True,3101,63,0.399549,1862,1239



=== Preview Joined Dataset Columns ===


,sleep_episode_id,participant_object_id,sleep_start_datetime,previous_day_feature_date,previous_day_daily_joined,good_sleep_label,previous_day_resting_hr_resting_heart_rate_mean,previous_day_resting_hr_error_mean,previous_day_resting_hr_record_count,previous_day_lightly_active_minutes_sum,previous_day_moderately_active_minutes_sum,previous_day_sedentary_minutes_sum,previous_day_very_active_minutes_sum,previous_day_steps_sum,previous_day_steps_record_count,previous_day_calories_sum,previous_day_calories_record_count
0,621e2e8e67b776a24055b564__20210524004000__2021...,621e2e8e67b776a24055b564,2021-05-24 00:40:00,2021-05-23,False,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,621e2e8e67b776a24055b564__20210524234830__2021...,621e2e8e67b776a24055b564,2021-05-24 23:48:30,2021-05-23,False,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,621e2e8e67b776a24055b564__20210525234630__2021...,621e2e8e67b776a24055b564,2021-05-25 23:46:30,2021-05-24,True,1,62.073070,6.787090,1.0,149.0,24.0,713.0,33.0,8833.0,685.0,2351.59,1440.0
3,621e2e8e67b776a24055b564__20210526232130__2021...,621e2e8e67b776a24055b564,2021-05-26 23:21:30,2021-05-25,True,1,62.121476,6.787088,1.0,132.0,25.0,704.0,31.0,9727.0,674.0,2332.08,1440.0
4,621e2e8e67b776a24055b564__20210527233700__2021...,621e2e8e67b776a24055b564,2021-05-27 23:37:00,2021-05-26,True,1,62.263999,6.787087,1.0,112.0,27.0,710.0,31.0,8253.0,577.0,2262.30,1440.0
5,621e2e8e67b776a24055b564__20210528235000__2021...,621e2e8e67b776a24055b564,2021-05-28 23:50:00,2021-05-27,True,1,62.368900,6.787087,1.0,133.0,21.0,622.0,37.0,9015.0,614.0,2325.10,1440.0
6,621e2e8e67b776a24055b564__20210530003000__2021...,621e2e8e67b776a24055b564,2021-05-30 00:30:00,2021-05-29,True,1,62.671748,6.787087,1.0,305.0,128.0,371.0,98.0,25526.0,863.0,3806.02,1440.0
7,621e2e8e67b776a24055b564__20210530232930__2021...,621e2e8e67b776a24055b564,2021-05-30 23:29:30,2021-05-29,True,1,62.671748,6.787087,1.0,305.0,128.0,371.0,98.0,25526.0,863.0,3806.02,1440.0
8,621e2e8e67b776a24055b564__20210531234230__2021...,621e2e8e67b776a24055b564,2021-05-31 23:42:30,2021-05-30,True,1,63.359720,6.787087,1.0,113.0,9.0,763.0,0.0,3796.0,403.0,1968.24,1440.0
9,621e2e8e67b776a24055b564__20210601235830__2021...,621e2e8e67b776a24055b564,2021-06-01 23:58:30,2021-05-31,True,1,63.121265,6.787087,1.0,149.0,23.0,655.0,34.0,9245.0,569.0,2300.02,1440.0



=== Validation Checks ===
non-numeric previous-day features: 0
previous-day feature columns: 11
previous-day daily feature validation passed


In [4]:
# Cell 3. MongoDB strict pre-sleep intraday aggregation utilities and small sample check
# 원하는 결과:
# - MongoDB 연결과 id 타입을 확인한다.
# - steps/calories/heart_rate를 sleep_start_date 00:00 <= timestamp < sleep_start_datetime 조건으로 조회한다.
# - 5개 sleep episode 샘플에서 pre-sleep 집계가 정상적으로 나오는지 확인한다.
# - 아직 전체 dataset 생성/저장은 하지 않는다.

from datetime import datetime, timedelta
from bson import ObjectId
from pymongo import MongoClient

MONGO_URI = "mongodb://localhost:27017"
MONGO_DB_NAME = "rais_anonymized"
MONGO_COLLECTION_NAME = "fitbit"

client = MongoClient(MONGO_URI)
fitbit_collection = client[MONGO_DB_NAME][MONGO_COLLECTION_NAME]

INTRADAY_TYPES = ["steps", "calories", "heart_rate"]

def resolve_mongo_participant_id(participant_id):
    """MongoDB fitbit.id is stored as ObjectId in this dataset."""
    if isinstance(participant_id, ObjectId):
        return participant_id
    return ObjectId(str(participant_id))

def format_mongo_datetime_for_type(timestamp, fitbit_type):
    if fitbit_type == "calories":
        return timestamp.strftime("%m/%d/%y %H:%M:%S")
    return timestamp.strftime("%Y-%m-%dT%H:%M:%S")

def build_datetime_filter_for_type(start_dt, end_dt, fitbit_type):
    return {
        "$gte": format_mongo_datetime_for_type(start_dt, fitbit_type),
        "$lt": format_mongo_datetime_for_type(end_dt, fitbit_type),
    }

def parse_intraday_datetime(value, fitbit_type):
    if pd.isna(value):
        return pd.NaT
    if fitbit_type == "calories":
        return pd.to_datetime(value, format="%m/%d/%y %H:%M:%S", errors="coerce")
    return pd.to_datetime(value, errors="coerce")

def load_intraday_records(participant_id, fitbit_type, start_dt, end_dt):
    mongo_id = resolve_mongo_participant_id(participant_id)
    datetime_filter = build_datetime_filter_for_type(start_dt, end_dt, fitbit_type)

    cursor = fitbit_collection.find(
        {
            "id": mongo_id,
            "type": fitbit_type,
            "data.dateTime": datetime_filter,
        },
        {
            "_id": 0,
            "id": 1,
            "type": 1,
            "data.dateTime": 1,
            "data.value": 1,
        },
    )

    rows = []
    for doc in cursor:
        data = doc.get("data", {})
        rows.append(
            {
                ID_COL: str(doc.get("id")),
                "fitbit_type": doc.get("type"),
                "timestamp_raw": data.get("dateTime"),
                "value_raw": data.get("value"),
            }
        )

    if not rows:
        return pd.DataFrame(
            columns=[ID_COL, "fitbit_type", "timestamp", "value", "confidence"]
        )

    records_df = pd.DataFrame(rows)
    records_df["timestamp"] = records_df["timestamp_raw"].apply(
        lambda x: parse_intraday_datetime(x, fitbit_type)
    )

    if fitbit_type == "heart_rate":
        records_df["value"] = records_df["value_raw"].apply(
            lambda x: x.get("bpm") if isinstance(x, dict) else np.nan
        )
        records_df["confidence"] = records_df["value_raw"].apply(
            lambda x: x.get("confidence") if isinstance(x, dict) else np.nan
        )
    else:
        records_df["value"] = pd.to_numeric(records_df["value_raw"], errors="coerce")
        records_df["confidence"] = np.nan

    records_df = records_df[
        (records_df["timestamp"] >= start_dt)
        & (records_df["timestamp"] < end_dt)
    ].copy()

    return records_df[[ID_COL, "fitbit_type", "timestamp", "value", "confidence"]]

def summarize_pre_sleep_intraday(records_df, fitbit_type, sleep_start_datetime):
    if fitbit_type == "steps":
        prefix = "steps_pre_sleep"
        empty = {
            f"{prefix}_sum": np.nan,
            f"{prefix}_record_count": 0,
            f"{prefix}_active_record_count": 0,
            f"{prefix}_last_3h_sum": np.nan,
            f"{prefix}_last_1h_sum": np.nan,
        }
        if records_df.empty:
            return empty

        last_3h_start = sleep_start_datetime - pd.Timedelta(hours=3)
        last_1h_start = sleep_start_datetime - pd.Timedelta(hours=1)

        return {
            f"{prefix}_sum": records_df["value"].sum(),
            f"{prefix}_record_count": int(records_df["value"].notna().sum()),
            f"{prefix}_active_record_count": int((records_df["value"].fillna(0) > 0).sum()),
            f"{prefix}_last_3h_sum": records_df.loc[
                records_df["timestamp"] >= last_3h_start, "value"
            ].sum(),
            f"{prefix}_last_1h_sum": records_df.loc[
                records_df["timestamp"] >= last_1h_start, "value"
            ].sum(),
        }

    if fitbit_type == "calories":
        prefix = "calories_pre_sleep"
        empty = {
            f"{prefix}_sum": np.nan,
            f"{prefix}_record_count": 0,
            f"{prefix}_last_3h_sum": np.nan,
            f"{prefix}_last_1h_sum": np.nan,
        }
        if records_df.empty:
            return empty

        last_3h_start = sleep_start_datetime - pd.Timedelta(hours=3)
        last_1h_start = sleep_start_datetime - pd.Timedelta(hours=1)

        return {
            f"{prefix}_sum": records_df["value"].sum(),
            f"{prefix}_record_count": int(records_df["value"].notna().sum()),
            f"{prefix}_last_3h_sum": records_df.loc[
                records_df["timestamp"] >= last_3h_start, "value"
            ].sum(),
            f"{prefix}_last_1h_sum": records_df.loc[
                records_df["timestamp"] >= last_1h_start, "value"
            ].sum(),
        }

    if fitbit_type == "heart_rate":
        prefix = "heart_rate_pre_sleep"
        empty = {
            f"{prefix}_mean": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_max": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_last_3h_mean": np.nan,
            f"{prefix}_last_1h_mean": np.nan,
            f"{prefix}_record_count": 0,
            f"{prefix}_mean_confidence": np.nan,
        }
        if records_df.empty:
            return empty

        last_3h_start = sleep_start_datetime - pd.Timedelta(hours=3)
        last_1h_start = sleep_start_datetime - pd.Timedelta(hours=1)

        values = records_df["value"].dropna()
        last_3h_values = records_df.loc[
            records_df["timestamp"] >= last_3h_start, "value"
        ].dropna()
        last_1h_values = records_df.loc[
            records_df["timestamp"] >= last_1h_start, "value"
        ].dropna()

        return {
            f"{prefix}_mean": values.mean(),
            f"{prefix}_std": values.std(ddof=0),
            f"{prefix}_min": values.min(),
            f"{prefix}_max": values.max(),
            f"{prefix}_median": values.median(),
            f"{prefix}_last_3h_mean": last_3h_values.mean(),
            f"{prefix}_last_1h_mean": last_1h_values.mean(),
            f"{prefix}_record_count": int(values.shape[0]),
            f"{prefix}_mean_confidence": records_df["confidence"].mean(),
        }

    raise ValueError(f"Unsupported fitbit_type: {fitbit_type}")

def aggregate_episode_pre_sleep_intraday(episode_row):
    sleep_start_datetime = episode_row["sleep_start_datetime"]
    window_start = episode_row["sleep_start_date"]
    window_end = sleep_start_datetime

    output = {
        "sleep_episode_id": episode_row["sleep_episode_id"],
        ID_COL: episode_row[ID_COL],
        "sleep_start_datetime": sleep_start_datetime,
        "pre_sleep_window_start": window_start,
        "pre_sleep_window_end": window_end,
        "pre_sleep_window_hours": (
            window_end - window_start
        ).total_seconds() / 3600,
    }

    for fitbit_type in INTRADAY_TYPES:
        records_df = load_intraday_records(
            participant_id=episode_row[ID_COL],
            fitbit_type=fitbit_type,
            start_dt=window_start,
            end_dt=window_end,
        )
        output.update(
            summarize_pre_sleep_intraday(
                records_df=records_df,
                fitbit_type=fitbit_type,
                sleep_start_datetime=sleep_start_datetime,
            )
        )

    return output

print("=== MongoDB Connection Check ===")
print("database:", MONGO_DB_NAME)
print("collection:", MONGO_COLLECTION_NAME)
print("estimated documents:", fitbit_collection.estimated_document_count())

print("\n=== MongoDB id Type Check ===")
id_type_rows = []
for fitbit_type in INTRADAY_TYPES:
    sample_doc = fitbit_collection.find_one(
        {"type": fitbit_type},
        {"_id": 0, "id": 1, "type": 1, "data.dateTime": 1},
    )
    id_value = sample_doc.get("id") if sample_doc else None
    id_type_rows.append(
        {
            "fitbit_type": fitbit_type,
            "sample_id": str(id_value),
            "python_type": type(id_value).__name__,
            "sample_datetime": sample_doc.get("data", {}).get("dateTime") if sample_doc else None,
        }
    )
display(pd.DataFrame(id_type_rows))

pre_sleep_design_df = pre_sleep_design_df.copy()
pre_sleep_design_df["pre_sleep_window_start"] = pre_sleep_design_df["sleep_start_date"]
pre_sleep_design_df["pre_sleep_window_end"] = pre_sleep_design_df["sleep_start_datetime"]
pre_sleep_design_df["pre_sleep_window_hours"] = (
    pre_sleep_design_df["pre_sleep_window_end"]
    - pre_sleep_design_df["pre_sleep_window_start"]
).dt.total_seconds() / 3600

sample_episodes = (
    pre_sleep_design_df
    .sort_values([ID_COL, "sleep_start_datetime"])
    .loc[lambda df: df["pre_sleep_window_hours"] >= 3]
    .head(5)
    .copy()
)

print("\n=== Sample Episodes For Intraday Aggregation ===")
display(
    sample_episodes[
        [
            "sleep_episode_id",
            ID_COL,
            "sleep_start_datetime",
            "sleep_start_date",
            "pre_sleep_window_hours",
            TARGET_COL,
        ]
    ]
)

sample_aggregation_rows = []
for _, episode_row in sample_episodes.iterrows():
    sample_aggregation_rows.append(
        aggregate_episode_pre_sleep_intraday(episode_row)
    )

sample_intraday_features_df = pd.DataFrame(sample_aggregation_rows)

print("\n=== Sample Pre-Sleep Intraday Aggregation ===")
display(sample_intraday_features_df)

record_count_cols = [
    col for col in sample_intraday_features_df.columns
    if col.endswith("_record_count")
]

print("\n=== Sample Record Count Summary ===")
display(sample_intraday_features_df[record_count_cols].describe().T)

print("\n=== Validation Checks ===")
print("sample episodes:", len(sample_intraday_features_df))
print("record count columns:", record_count_cols)
print("rows with any intraday records:", int((sample_intraday_features_df[record_count_cols].sum(axis=1) > 0).sum()))

if (sample_intraday_features_df[record_count_cols].sum(axis=1) > 0).all():
    print("sample intraday aggregation validation passed")
else:
    print("some sample episodes have no intraday records; inspect before full run")

=== MongoDB Connection Check ===
database: rais_anonymized
collection: fitbit
estimated documents: 71284346

=== MongoDB id Type Check ===


,fitbit_type,sample_id,python_type,sample_datetime
0,steps,621e2e8e67b776a24055b564,ObjectId,2021-05-24T00:15:00
1,calories,621e2e8e67b776a24055b564,ObjectId,05/24/21 00:00:00
2,heart_rate,621e2e8e67b776a24055b564,ObjectId,2021-05-24T00:00:16



=== Sample Episodes For Intraday Aggregation ===


,sleep_episode_id,participant_object_id,sleep_start_datetime,sleep_start_date,pre_sleep_window_hours,good_sleep_label
1,621e2e8e67b776a24055b564__20210524234830__2021...,621e2e8e67b776a24055b564,2021-05-24 23:48:30,2021-05-24,23.808333,1
2,621e2e8e67b776a24055b564__20210525234630__2021...,621e2e8e67b776a24055b564,2021-05-25 23:46:30,2021-05-25,23.775000,1
3,621e2e8e67b776a24055b564__20210526232130__2021...,621e2e8e67b776a24055b564,2021-05-26 23:21:30,2021-05-26,23.358333,1
4,621e2e8e67b776a24055b564__20210527233700__2021...,621e2e8e67b776a24055b564,2021-05-27 23:37:00,2021-05-27,23.616667,1
5,621e2e8e67b776a24055b564__20210528235000__2021...,621e2e8e67b776a24055b564,2021-05-28 23:50:00,2021-05-28,23.833333,1



=== Sample Pre-Sleep Intraday Aggregation ===


,sleep_episode_id,participant_object_id,sleep_start_datetime,pre_sleep_window_start,pre_sleep_window_end,pre_sleep_window_hours,steps_pre_sleep_sum,steps_pre_sleep_record_count,steps_pre_sleep_active_record_count,steps_pre_sleep_last_3h_sum,...,calories_pre_sleep_last_1h_sum,heart_rate_pre_sleep_mean,heart_rate_pre_sleep_std,heart_rate_pre_sleep_min,heart_rate_pre_sleep_max,heart_rate_pre_sleep_median,heart_rate_pre_sleep_last_3h_mean,heart_rate_pre_sleep_last_1h_mean,heart_rate_pre_sleep_record_count,heart_rate_pre_sleep_mean_confidence
0,621e2e8e67b776a24055b564__20210524234830__2021...,621e2e8e67b776a24055b564,2021-05-24 23:48:30,2021-05-24,2021-05-24 23:48:30,23.808333,8823,679,198,120,...,95.08,71.677430,13.468821,32,144,70.0,75.044199,76.013566,12481,1.813877
1,621e2e8e67b776a24055b564__20210525234630__2021...,621e2e8e67b776a24055b564,2021-05-25 23:46:30,2021-05-25,2021-05-25 23:46:30,23.775000,9720,672,185,136,...,81.73,70.633747,13.622614,32,143,69.0,73.439593,69.444640,12576,1.840808
2,621e2e8e67b776a24055b564__20210526232130__2021...,621e2e8e67b776a24055b564,2021-05-26 23:21:30,2021-05-26,2021-05-26 23:21:30,23.358333,8211,572,166,152,...,75.70,71.978559,14.139740,36,127,70.0,78.491515,68.859848,12173,1.774747
3,621e2e8e67b776a24055b564__20210527233700__2021...,621e2e8e67b776a24055b564,2021-05-27 23:37:00,2021-05-27,2021-05-27 23:37:00,23.616667,9007,612,184,178,...,65.51,71.826641,14.779616,49,131,70.0,84.908057,71.833333,11854,1.775013
4,621e2e8e67b776a24055b564__20210528235000__2021...,621e2e8e67b776a24055b564,2021-05-28 23:50:00,2021-05-28,2021-05-28 23:50:00,23.833333,12944,760,216,222,...,77.46,74.505052,16.847051,30,149,75.0,70.382740,66.724346,12173,1.814672



=== Sample Record Count Summary ===


,count,mean,std,min,25%,50%,75%,max
steps_pre_sleep_record_count,5.0,659.0,71.672868,572.0,612.0,672.0,679.0,760.0
steps_pre_sleep_active_record_count,5.0,189.8,18.552628,166.0,184.0,185.0,198.0,216.0
calories_pre_sleep_record_count,5.0,1421.0,11.811012,1402.0,1417.0,1427.0,1429.0,1430.0
heart_rate_pre_sleep_record_count,5.0,12251.4,286.487871,11854.0,12173.0,12173.0,12481.0,12576.0



=== Validation Checks ===
sample episodes: 5
record count columns: ['steps_pre_sleep_record_count', 'steps_pre_sleep_active_record_count', 'calories_pre_sleep_record_count', 'heart_rate_pre_sleep_record_count']
rows with any intraday records: 5
sample intraday aggregation validation passed


In [8]:
# Cell 4. MongoDB server-side strict pre-sleep intraday aggregation
# 원하는 결과:
# - MongoDB aggregation pipeline으로 raw intraday rows를 Python으로 가져오지 않고 서버에서 집계한다.
# - steps/calories/heart_rate pre-sleep feature를 전체 sleep episode에 대해 생성한다.
# - 이전 episode-wise raw fetch 방식보다 속도가 개선되는지 확인한다.
# - 아직 파일 저장은 하지 않는다.

import time

RUN_SERVER_SIDE_INTRADAY_AGGREGATION = True

def aggregate_steps_server_side(participant_id, start_dt, end_dt, sleep_start_datetime):
    mongo_id = resolve_mongo_participant_id(participant_id)

    start_str = format_mongo_datetime_for_type(start_dt, "steps")
    end_str = format_mongo_datetime_for_type(end_dt, "steps")
    last_3h_str = format_mongo_datetime_for_type(sleep_start_datetime - pd.Timedelta(hours=3), "steps")
    last_1h_str = format_mongo_datetime_for_type(sleep_start_datetime - pd.Timedelta(hours=1), "steps")

    pipeline = [
        {
            "$match": {
                "id": mongo_id,
                "type": "steps",
                "data.dateTime": {"$gte": start_str, "$lt": end_str},
            }
        },
        {
            "$project": {
                "_id": 0,
                "dateTime": "$data.dateTime",
                "value": {"$toDouble": "$data.value"},
            }
        },
        {
            "$group": {
                "_id": None,
                "steps_pre_sleep_sum": {"$sum": "$value"},
                "steps_pre_sleep_record_count": {"$sum": 1},
                "steps_pre_sleep_active_record_count": {
                    "$sum": {"$cond": [{"$gt": ["$value", 0]}, 1, 0]}
                },
                "steps_pre_sleep_last_3h_sum": {
                    "$sum": {
                        "$cond": [
                            {"$gte": ["$dateTime", last_3h_str]},
                            "$value",
                            0,
                        ]
                    }
                },
                "steps_pre_sleep_last_1h_sum": {
                    "$sum": {
                        "$cond": [
                            {"$gte": ["$dateTime", last_1h_str]},
                            "$value",
                            0,
                        ]
                    }
                },
            }
        },
    ]

    result = list(fitbit_collection.aggregate(pipeline, allowDiskUse=False))
    if not result:
        return {
            "steps_pre_sleep_sum": np.nan,
            "steps_pre_sleep_record_count": 0,
            "steps_pre_sleep_active_record_count": 0,
            "steps_pre_sleep_last_3h_sum": np.nan,
            "steps_pre_sleep_last_1h_sum": np.nan,
        }

    row = result[0]
    row.pop("_id", None)
    return row

def aggregate_calories_server_side(participant_id, start_dt, end_dt, sleep_start_datetime):
    mongo_id = resolve_mongo_participant_id(participant_id)

    start_str = format_mongo_datetime_for_type(start_dt, "calories")
    end_str = format_mongo_datetime_for_type(end_dt, "calories")
    last_3h_str = format_mongo_datetime_for_type(sleep_start_datetime - pd.Timedelta(hours=3), "calories")
    last_1h_str = format_mongo_datetime_for_type(sleep_start_datetime - pd.Timedelta(hours=1), "calories")

    pipeline = [
        {
            "$match": {
                "id": mongo_id,
                "type": "calories",
                "data.dateTime": {"$gte": start_str, "$lt": end_str},
            }
        },
        {
            "$project": {
                "_id": 0,
                "dateTime": "$data.dateTime",
                "value": {"$toDouble": "$data.value"},
            }
        },
        {
            "$group": {
                "_id": None,
                "calories_pre_sleep_sum": {"$sum": "$value"},
                "calories_pre_sleep_record_count": {"$sum": 1},
                "calories_pre_sleep_last_3h_sum": {
                    "$sum": {
                        "$cond": [
                            {"$gte": ["$dateTime", last_3h_str]},
                            "$value",
                            0,
                        ]
                    }
                },
                "calories_pre_sleep_last_1h_sum": {
                    "$sum": {
                        "$cond": [
                            {"$gte": ["$dateTime", last_1h_str]},
                            "$value",
                            0,
                        ]
                    }
                },
            }
        },
    ]

    result = list(fitbit_collection.aggregate(pipeline, allowDiskUse=False))
    if not result:
        return {
            "calories_pre_sleep_sum": np.nan,
            "calories_pre_sleep_record_count": 0,
            "calories_pre_sleep_last_3h_sum": np.nan,
            "calories_pre_sleep_last_1h_sum": np.nan,
        }

    row = result[0]
    row.pop("_id", None)
    return row

def aggregate_heart_rate_server_side(participant_id, start_dt, end_dt, sleep_start_datetime):
    mongo_id = resolve_mongo_participant_id(participant_id)

    start_str = format_mongo_datetime_for_type(start_dt, "heart_rate")
    end_str = format_mongo_datetime_for_type(end_dt, "heart_rate")
    last_3h_str = format_mongo_datetime_for_type(sleep_start_datetime - pd.Timedelta(hours=3), "heart_rate")
    last_1h_str = format_mongo_datetime_for_type(sleep_start_datetime - pd.Timedelta(hours=1), "heart_rate")

    pipeline = [
        {
            "$match": {
                "id": mongo_id,
                "type": "heart_rate",
                "data.dateTime": {"$gte": start_str, "$lt": end_str},
            }
        },
        {
            "$project": {
                "_id": 0,
                "dateTime": "$data.dateTime",
                "bpm": {"$toDouble": "$data.value.bpm"},
                "confidence": {"$toDouble": "$data.value.confidence"},
            }
        },
        {
            "$group": {
                "_id": None,
                "heart_rate_pre_sleep_mean": {"$avg": "$bpm"},
                "heart_rate_pre_sleep_std": {"$stdDevPop": "$bpm"},
                "heart_rate_pre_sleep_min": {"$min": "$bpm"},
                "heart_rate_pre_sleep_max": {"$max": "$bpm"},
                "heart_rate_pre_sleep_record_count": {"$sum": 1},
                "heart_rate_pre_sleep_mean_confidence": {"$avg": "$confidence"},
                "heart_rate_pre_sleep_last_3h_mean": {
                    "$avg": {
                        "$cond": [
                            {"$gte": ["$dateTime", last_3h_str]},
                            "$bpm",
                            "$$REMOVE",
                        ]
                    }
                },
                "heart_rate_pre_sleep_last_1h_mean": {
                    "$avg": {
                        "$cond": [
                            {"$gte": ["$dateTime", last_1h_str]},
                            "$bpm",
                            "$$REMOVE",
                        ]
                    }
                },
                "heart_rate_values": {"$push": "$bpm"},
            }
        },
        {
            "$project": {
                "_id": 0,
                "heart_rate_pre_sleep_mean": 1,
                "heart_rate_pre_sleep_std": 1,
                "heart_rate_pre_sleep_min": 1,
                "heart_rate_pre_sleep_max": 1,
                "heart_rate_pre_sleep_record_count": 1,
                "heart_rate_pre_sleep_mean_confidence": 1,
                "heart_rate_pre_sleep_last_3h_mean": 1,
                "heart_rate_pre_sleep_last_1h_mean": 1,
                "heart_rate_pre_sleep_median": {
                    "$arrayElemAt": [
                        {
                            "$sortArray": {
                                "input": "$heart_rate_values",
                                "sortBy": 1,
                            }
                        },
                        {
                            "$floor": {
                                "$divide": [
                                    {"$size": "$heart_rate_values"},
                                    2,
                                ]
                            }
                        },
                    ]
                },
            }
        },
    ]

    result = list(fitbit_collection.aggregate(pipeline, allowDiskUse=False))
    if not result:
        return {
            "heart_rate_pre_sleep_mean": np.nan,
            "heart_rate_pre_sleep_std": np.nan,
            "heart_rate_pre_sleep_min": np.nan,
            "heart_rate_pre_sleep_max": np.nan,
            "heart_rate_pre_sleep_median": np.nan,
            "heart_rate_pre_sleep_last_3h_mean": np.nan,
            "heart_rate_pre_sleep_last_1h_mean": np.nan,
            "heart_rate_pre_sleep_record_count": 0,
            "heart_rate_pre_sleep_mean_confidence": np.nan,
        }

    return result[0]

def aggregate_episode_pre_sleep_intraday_server_side(episode_row):
    sleep_start_datetime = episode_row["sleep_start_datetime"]
    window_start = episode_row["sleep_start_date"]
    window_end = sleep_start_datetime

    output = {
        "sleep_episode_id": episode_row["sleep_episode_id"],
        ID_COL: episode_row[ID_COL],
        "sleep_start_datetime": sleep_start_datetime,
        "pre_sleep_window_start": window_start,
        "pre_sleep_window_end": window_end,
        "pre_sleep_window_hours": (
            window_end - window_start
        ).total_seconds() / 3600,
    }

    output.update(
        aggregate_steps_server_side(
            participant_id=episode_row[ID_COL],
            start_dt=window_start,
            end_dt=window_end,
            sleep_start_datetime=sleep_start_datetime,
        )
    )

    output.update(
        aggregate_calories_server_side(
            participant_id=episode_row[ID_COL],
            start_dt=window_start,
            end_dt=window_end,
            sleep_start_datetime=sleep_start_datetime,
        )
    )

    output.update(
        aggregate_heart_rate_server_side(
            participant_id=episode_row[ID_COL],
            start_dt=window_start,
            end_dt=window_end,
            sleep_start_datetime=sleep_start_datetime,
        )
    )

    return output

if not RUN_SERVER_SIDE_INTRADAY_AGGREGATION:
    print("RUN_SERVER_SIDE_INTRADAY_AGGREGATION is False.")
else:
    if "pre_sleep_window_hours" not in pre_sleep_design_df.columns:
        pre_sleep_design_df = pre_sleep_design_df.copy()
        pre_sleep_design_df["pre_sleep_window_start"] = pre_sleep_design_df["sleep_start_date"]
        pre_sleep_design_df["pre_sleep_window_end"] = pre_sleep_design_df["sleep_start_datetime"]
        pre_sleep_design_df["pre_sleep_window_hours"] = (
            pre_sleep_design_df["pre_sleep_window_end"]
            - pre_sleep_design_df["pre_sleep_window_start"]
        ).dt.total_seconds() / 3600

    episodes_for_intraday = (
        pre_sleep_design_df
        .sort_values([ID_COL, "sleep_start_datetime"])
        .reset_index(drop=True)
        .copy()
    )

    aggregation_start_time = time.time()
    intraday_rows = []

    total_episodes = len(episodes_for_intraday)

    print("=== Server-Side Intraday Aggregation Started ===")
    print("episodes:", total_episodes)
    print("strategy: MongoDB aggregation returns summary rows, not raw intraday rows")
    print("expected index: { id: 1, type: 1, data.dateTime: 1 }")

    for row_index, episode_row in episodes_for_intraday.iterrows():
        intraday_rows.append(
            aggregate_episode_pre_sleep_intraday_server_side(episode_row)
        )

        completed = row_index + 1
        if completed == 1 or completed % 100 == 0 or completed == total_episodes:
            elapsed_seconds = time.time() - aggregation_start_time
            seconds_per_episode = elapsed_seconds / completed
            eta_seconds = seconds_per_episode * (total_episodes - completed)

            print(
                f"{completed}/{total_episodes} episodes | "
                f"elapsed={elapsed_seconds/60:.1f} min | "
                f"avg={seconds_per_episode:.2f} sec/episode | "
                f"eta={eta_seconds/60:.1f} min"
            )

            if completed == 100 and eta_seconds / 60 > 90:
                print("WARNING: ETA is still high after 100 episodes. Consider interrupting.")

    intraday_features_df = pd.DataFrame(intraday_rows)

    intraday_feature_cols = [
        col for col in intraday_features_df.columns
        if col not in [
            "sleep_episode_id",
            ID_COL,
            "sleep_start_datetime",
            "pre_sleep_window_start",
            "pre_sleep_window_end",
            "pre_sleep_window_hours",
        ]
    ]

    intraday_record_count_cols = [
        col for col in intraday_features_df.columns
        if col.endswith("_record_count")
    ]

    intraday_coverage_summary = pd.DataFrame(
        [
            {"metric": "episodes", "value": len(intraday_features_df)},
            {
                "metric": "steps_rows_with_records",
                "value": int((intraday_features_df["steps_pre_sleep_record_count"] > 0).sum()),
            },
            {
                "metric": "calories_rows_with_records",
                "value": int((intraday_features_df["calories_pre_sleep_record_count"] > 0).sum()),
            },
            {
                "metric": "heart_rate_rows_with_records",
                "value": int((intraday_features_df["heart_rate_pre_sleep_record_count"] > 0).sum()),
            },
            {
                "metric": "steps_record_coverage",
                "value": float((intraday_features_df["steps_pre_sleep_record_count"] > 0).mean()),
            },
            {
                "metric": "calories_record_coverage",
                "value": float((intraday_features_df["calories_pre_sleep_record_count"] > 0).mean()),
            },
            {
                "metric": "heart_rate_record_coverage",
                "value": float((intraday_features_df["heart_rate_pre_sleep_record_count"] > 0).mean()),
            },
        ]
    )

    intraday_record_count_summary = (
        intraday_features_df[intraday_record_count_cols]
        .describe()
        .T
        .reset_index()
        .rename(columns={"index": "record_count_feature"})
    )

    intraday_missing_summary = (
        intraday_features_df[intraday_feature_cols]
        .isna()
        .mean()
        .rename("missing_rate")
        .reset_index()
        .rename(columns={"index": "feature"})
        .sort_values(["missing_rate", "feature"], ascending=[False, True])
    )

    elapsed_total_seconds = time.time() - aggregation_start_time

    print("\n=== Server-Side Intraday Aggregation Finished ===")
    print(f"elapsed minutes: {elapsed_total_seconds / 60:.2f}")
    print("intraday_features_df shape:", intraday_features_df.shape)
    print("intraday feature columns:", len(intraday_feature_cols))

    print("\n=== Intraday Coverage Summary ===")
    display(intraday_coverage_summary)

    print("\n=== Intraday Record Count Summary ===")
    display(intraday_record_count_summary)

    print("\n=== Intraday Missing Summary ===")
    display(intraday_missing_summary)

    print("\n=== Intraday Feature Preview ===")
    display(intraday_features_df.head(10))

    print("\n=== Validation Checks ===")
    print("unique sleep_episode_id:", intraday_features_df["sleep_episode_id"].nunique())
    print("expected episodes:", len(pre_sleep_design_df))
    print(
        "duplicate sleep_episode_id rows:",
        len(intraday_features_df) - intraday_features_df["sleep_episode_id"].nunique(),
    )

    if intraday_features_df["sleep_episode_id"].nunique() == len(pre_sleep_design_df):
        print("server-side intraday aggregation validation passed")
    else:
        print("episode count mismatch; inspect before saving")

=== Server-Side Intraday Aggregation Started ===
episodes: 3551
strategy: MongoDB aggregation returns summary rows, not raw intraday rows
expected index: { id: 1, type: 1, data.dateTime: 1 }
1/3551 episodes | elapsed=0.0 min | avg=0.10 sec/episode | eta=5.7 min
100/3551 episodes | elapsed=0.1 min | avg=0.06 sec/episode | eta=3.2 min
200/3551 episodes | elapsed=0.2 min | avg=0.06 sec/episode | eta=3.3 min
300/3551 episodes | elapsed=0.3 min | avg=0.06 sec/episode | eta=3.2 min
400/3551 episodes | elapsed=0.4 min | avg=0.06 sec/episode | eta=3.1 min
500/3551 episodes | elapsed=0.5 min | avg=0.06 sec/episode | eta=2.8 min
600/3551 episodes | elapsed=0.6 min | avg=0.06 sec/episode | eta=2.8 min
700/3551 episodes | elapsed=0.6 min | avg=0.05 sec/episode | eta=2.6 min
800/3551 episodes | elapsed=0.7 min | avg=0.05 sec/episode | eta=2.4 min
900/3551 episodes | elapsed=0.7 min | avg=0.05 sec/episode | eta=2.2 min
1000/3551 episodes | elapsed=0.8 min | avg=0.05 sec/episode | eta=2.0 min
1100/35

,metric,value
0,episodes,3551.000000
1,steps_rows_with_records,3381.000000
2,calories_rows_with_records,3534.000000
3,heart_rate_rows_with_records,3451.000000
4,steps_record_coverage,0.952126
5,calories_record_coverage,0.995213
6,heart_rate_record_coverage,0.971839



=== Intraday Record Count Summary ===


,record_count_feature,count,mean,std,min,25%,50%,75%,max
0,steps_pre_sleep_record_count,3551.0,285.475077,312.473843,0.0,22.0,99.0,585.0,1236.0
1,steps_pre_sleep_active_record_count,3551.0,100.986483,124.260385,0.0,5.0,28.0,185.0,654.0
2,calories_pre_sleep_record_count,3551.0,613.435370,620.931849,0.0,72.0,199.0,1368.5,2762.0
3,heart_rate_pre_sleep_record_count,3551.0,4709.425796,5049.106620,0.0,495.0,1525.0,10817.0,23287.0



=== Intraday Missing Summary ===


,feature,missing_rate
4,steps_pre_sleep_last_1h_sum,0.047874
3,steps_pre_sleep_last_3h_sum,0.047874
0,steps_pre_sleep_sum,0.047874
16,heart_rate_pre_sleep_last_1h_mean,0.042523
15,heart_rate_pre_sleep_last_3h_mean,0.032948
12,heart_rate_pre_sleep_max,0.028161
9,heart_rate_pre_sleep_mean,0.028161
14,heart_rate_pre_sleep_mean_confidence,0.028161
17,heart_rate_pre_sleep_median,0.028161
11,heart_rate_pre_sleep_min,0.028161



=== Intraday Feature Preview ===


,sleep_episode_id,participant_object_id,sleep_start_datetime,pre_sleep_window_start,pre_sleep_window_end,pre_sleep_window_hours,steps_pre_sleep_sum,steps_pre_sleep_record_count,steps_pre_sleep_active_record_count,steps_pre_sleep_last_3h_sum,...,calories_pre_sleep_last_1h_sum,heart_rate_pre_sleep_mean,heart_rate_pre_sleep_std,heart_rate_pre_sleep_min,heart_rate_pre_sleep_max,heart_rate_pre_sleep_record_count,heart_rate_pre_sleep_mean_confidence,heart_rate_pre_sleep_last_3h_mean,heart_rate_pre_sleep_last_1h_mean,heart_rate_pre_sleep_median
0,621e2e8e67b776a24055b564__20210524004000__2021...,621e2e8e67b776a24055b564,2021-05-24 00:40:00,2021-05-24,2021-05-24 00:40:00,0.666667,82.0,16,8,82.0,...,59.69,68.162921,7.322919,59.0,94.0,356,1.584270,68.162921,68.162921,66.0
1,621e2e8e67b776a24055b564__20210524234830__2021...,621e2e8e67b776a24055b564,2021-05-24 23:48:30,2021-05-24,2021-05-24 23:48:30,23.808333,8823.0,679,198,120.0,...,95.08,71.677430,13.468821,32.0,144.0,12481,1.813877,75.044199,76.013566,70.0
2,621e2e8e67b776a24055b564__20210525234630__2021...,621e2e8e67b776a24055b564,2021-05-25 23:46:30,2021-05-25,2021-05-25 23:46:30,23.775000,9720.0,672,185,136.0,...,81.73,70.633747,13.622614,32.0,143.0,12576,1.840808,73.439593,69.444640,69.0
3,621e2e8e67b776a24055b564__20210526232130__2021...,621e2e8e67b776a24055b564,2021-05-26 23:21:30,2021-05-26,2021-05-26 23:21:30,23.358333,8211.0,572,166,152.0,...,75.70,71.978559,14.139740,36.0,127.0,12173,1.774747,78.491515,68.859848,70.0
4,621e2e8e67b776a24055b564__20210527233700__2021...,621e2e8e67b776a24055b564,2021-05-27 23:37:00,2021-05-27,2021-05-27 23:37:00,23.616667,9007.0,612,184,178.0,...,65.51,71.826641,14.779616,49.0,131.0,11854,1.775013,84.908057,71.833333,70.0
5,621e2e8e67b776a24055b564__20210528235000__2021...,621e2e8e67b776a24055b564,2021-05-28 23:50:00,2021-05-28,2021-05-28 23:50:00,23.833333,12944.0,760,216,222.0,...,77.46,74.505052,16.847051,30.0,149.0,12173,1.814672,70.382740,66.724346,75.0
6,621e2e8e67b776a24055b564__20210530003000__2021...,621e2e8e67b776a24055b564,2021-05-30 00:30:00,2021-05-30,2021-05-30 00:30:00,0.500000,23.0,14,3,23.0,...,41.69,74.650794,7.468790,32.0,90.0,252,1.603175,74.650794,74.650794,74.0
7,621e2e8e67b776a24055b564__20210530232930__2021...,621e2e8e67b776a24055b564,2021-05-30 23:29:30,2021-05-30,2021-05-30 23:29:30,23.491667,3796.0,402,119,259.0,...,102.94,68.589150,11.629758,32.0,121.0,9770,2.011668,70.266055,74.764065,66.0
8,621e2e8e67b776a24055b564__20210531234230__2021...,621e2e8e67b776a24055b564,2021-05-31 23:42:30,2021-05-31,2021-05-31 23:42:30,23.708333,9228.0,567,200,510.0,...,74.28,69.420099,13.171650,30.0,126.0,12478,1.835230,71.535428,66.166969,66.0
9,621e2e8e67b776a24055b564__20210601235830__2021...,621e2e8e67b776a24055b564,2021-06-01 23:58:30,2021-06-01,2021-06-01 23:58:30,23.975000,8422.0,600,161,159.0,...,79.24,69.309680,13.376769,42.0,140.0,12345,1.814338,78.249533,75.366025,67.0



=== Validation Checks ===
unique sleep_episode_id: 3551
expected episodes: 3551
duplicate sleep_episode_id rows: 0
server-side intraday aggregation validation passed


In [9]:
# Cell 5. Validate server-side aggregation against raw-fetch aggregation
# 원하는 결과:
# - server-side aggregation 결과와 raw-fetch aggregation 결과를 샘플 episode에서 직접 비교한다.
# - MongoDB 서버 집계 방식이 기존 정확한 raw-fetch 방식과 같은 값을 내는지 확인한다.
# - 차이가 있으면 어떤 feature에서 얼마나 차이나는지 표시한다.
# - 아직 파일 저장은 하지 않는다.

VALIDATION_SAMPLE_SIZE = 10
NUMERIC_TOLERANCE = 1e-6

validation_sample_episodes = (
    pre_sleep_design_df
    .sort_values([ID_COL, "sleep_start_datetime"])
    .loc[lambda df: df["pre_sleep_window_hours"] >= 3]
    .sample(
        n=min(VALIDATION_SAMPLE_SIZE, len(pre_sleep_design_df)),
        random_state=2026,
    )
    .copy()
)

raw_validation_rows = []
server_validation_rows = []

print("=== Server-Side vs Raw-Fetch Validation Started ===")
print("sample episodes:", len(validation_sample_episodes))
print("numeric tolerance:", NUMERIC_TOLERANCE)

for _, episode_row in validation_sample_episodes.iterrows():
    raw_validation_rows.append(
        aggregate_episode_pre_sleep_intraday(episode_row)
    )
    server_validation_rows.append(
        aggregate_episode_pre_sleep_intraday_server_side(episode_row)
    )

raw_validation_df = pd.DataFrame(raw_validation_rows)
server_validation_df = pd.DataFrame(server_validation_rows)

comparison_feature_cols = [
    col for col in server_validation_df.columns
    if col not in [
        "sleep_episode_id",
        ID_COL,
        "sleep_start_datetime",
        "pre_sleep_window_start",
        "pre_sleep_window_end",
        "pre_sleep_window_hours",
    ]
]

comparison_rows = []

for feature in comparison_feature_cols:
    raw_values = pd.to_numeric(raw_validation_df[feature], errors="coerce")
    server_values = pd.to_numeric(server_validation_df[feature], errors="coerce")

    both_missing = raw_values.isna() & server_values.isna()
    absolute_diff = (raw_values - server_values).abs()

    mismatched = ~(both_missing | (absolute_diff <= NUMERIC_TOLERANCE))

    comparison_rows.append(
        {
            "feature": feature,
            "max_abs_diff": absolute_diff.max(skipna=True),
            "mismatch_rows": int(mismatched.sum()),
            "raw_missing_rows": int(raw_values.isna().sum()),
            "server_missing_rows": int(server_values.isna().sum()),
        }
    )

validation_comparison_df = pd.DataFrame(comparison_rows).sort_values(
    ["mismatch_rows", "max_abs_diff"],
    ascending=[False, False],
)

validation_detail_df = raw_validation_df[
    ["sleep_episode_id", ID_COL, "sleep_start_datetime"]
].copy()

for feature in comparison_feature_cols:
    validation_detail_df[f"raw__{feature}"] = raw_validation_df[feature]
    validation_detail_df[f"server__{feature}"] = server_validation_df[feature]
    validation_detail_df[f"abs_diff__{feature}"] = (
        pd.to_numeric(raw_validation_df[feature], errors="coerce")
        - pd.to_numeric(server_validation_df[feature], errors="coerce")
    ).abs()

problem_features = validation_comparison_df[
    validation_comparison_df["mismatch_rows"] > 0
]

print("\n=== Validation Comparison Summary ===")
display(validation_comparison_df)

print("\n=== Problem Features ===")
display(problem_features)

print("\n=== Validation Sample Episode IDs ===")
display(
    validation_sample_episodes[
        ["sleep_episode_id", ID_COL, "sleep_start_datetime", "pre_sleep_window_hours", TARGET_COL]
    ].reset_index(drop=True)
)

print("\n=== Validation Checks ===")
print("features compared:", len(comparison_feature_cols))
print("problem features:", len(problem_features))
print("total mismatch rows:", int(validation_comparison_df["mismatch_rows"].sum()))

if len(problem_features) == 0:
    print("server-side aggregation matches raw-fetch aggregation within tolerance")
else:
    print("server-side aggregation mismatch detected; inspect validation_detail_df before saving")

=== Server-Side vs Raw-Fetch Validation Started ===
sample episodes: 10
numeric tolerance: 1e-06

=== Validation Comparison Summary ===


,feature,max_abs_diff,mismatch_rows,raw_missing_rows,server_missing_rows
5,calories_pre_sleep_sum,4.547474e-13,0,0,0
10,heart_rate_pre_sleep_std,1.278977e-13,0,1,1
7,calories_pre_sleep_last_3h_sum,5.684342e-14,0,0,0
8,calories_pre_sleep_last_1h_sum,2.842171e-14,0,0,0
0,steps_pre_sleep_sum,0.000000e+00,0,1,1
1,steps_pre_sleep_record_count,0.000000e+00,0,0,0
2,steps_pre_sleep_active_record_count,0.000000e+00,0,0,0
3,steps_pre_sleep_last_3h_sum,0.000000e+00,0,1,1
4,steps_pre_sleep_last_1h_sum,0.000000e+00,0,1,1
6,calories_pre_sleep_record_count,0.000000e+00,0,0,0



=== Problem Features ===


,feature,max_abs_diff,mismatch_rows,raw_missing_rows,server_missing_rows



=== Validation Sample Episode IDs ===


,sleep_episode_id,participant_object_id,sleep_start_datetime,pre_sleep_window_hours,good_sleep_label
0,621e351a67b776a240f6204b__20210624041000__2021...,621e351a67b776a240f6204b,2021-06-24 04:10:00,4.166667,0
1,621e36f967b776a240e5e7c9__20210704232400__2021...,621e36f967b776a240e5e7c9,2021-07-04 23:24:00,23.400000,1
2,621e2ed667b776a24085d8d1__20210601231400__2021...,621e2ed667b776a24085d8d1,2021-06-01 23:14:00,23.233333,0
3,621e337667b776a240ce78ab__20210606234600__2021...,621e337667b776a240ce78ab,2021-06-06 23:46:00,23.766667,1
4,621e324e67b776a2400191cb__20211231031730__2021...,621e324e67b776a2400191cb,2021-12-31 03:17:30,3.291667,0
5,621e33ed67b776a2401cf5f7__20210715053600__2021...,621e33ed67b776a2401cf5f7,2021-07-15 05:36:00,5.600000,0
6,621e346f67b776a24081744f__20210703234500__2021...,621e346f67b776a24081744f,2021-07-03 23:45:00,23.750000,0
7,621e2f7a67b776a240f14425__20210718232930__2021...,621e2f7a67b776a240f14425,2021-07-18 23:29:30,23.491667,0
8,621e346f67b776a24081744f__20210611212200__2021...,621e346f67b776a24081744f,2021-06-11 21:22:00,21.366667,1
9,621e2ed667b776a24085d8d1__20210526215400__2021...,621e2ed667b776a24085d8d1,2021-05-26 21:54:00,21.900000,1



=== Validation Checks ===
features compared: 18
problem features: 0
total mismatch rows: 0
server-side aggregation matches raw-fetch aggregation within tolerance


In [10]:
# Cell 6. Build and save Design C Stage 1 pre-sleep forecasting dataset
# 원하는 결과:
# - sleep episode index + previous-day daily features + strict pre-sleep intraday features를 결합한다.
# - prediction-time timing/calendar features를 추가한다.
# - 결측 indicator를 만들고 numeric feature list를 저장한다.
# - Stage 1 Design C dataset과 metadata를 저장한다.
# - log/YYYY-MM-DD.md를 업데이트한다.

from datetime import datetime
import json

DESIGN_C_OUTPUT_DIR = PROCESSED_DIR / "pre_sleep_forecasting" / "design_c_stage1"
DESIGN_C_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STAGE1_DATASET_PATH = DESIGN_C_OUTPUT_DIR / "pre_sleep_design_c_stage1_dataset.csv"
STAGE1_FEATURE_COLUMNS_PATH = DESIGN_C_OUTPUT_DIR / "pre_sleep_design_c_stage1_feature_columns.csv"
STAGE1_MISSING_SUMMARY_PATH = DESIGN_C_OUTPUT_DIR / "pre_sleep_design_c_stage1_missing_summary.csv"
STAGE1_METADATA_PATH = DESIGN_C_OUTPUT_DIR / "metadata.json"

LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

base_episode_cols = [
    "sleep_episode_id",
    ID_COL,
    "calendar_date",
    "sleep_start_datetime",
    "sleep_end_datetime",
    "sleep_start_date",
    "sleep_end_date",
    "prediction_cutoff_datetime",
    TARGET_COL,
    "cross_midnight",
]

previous_day_keep_cols = [
    "sleep_episode_id",
    "previous_day_feature_date",
    "previous_day_daily_joined",
] + previous_day_feature_cols

intraday_keep_cols = [
    col for col in intraday_features_df.columns
    if col not in [ID_COL, "sleep_start_datetime"]
]

design_c_stage1_df = (
    pre_sleep_design_df[base_episode_cols + previous_day_keep_cols[1:]]
    .merge(
        intraday_features_df[intraday_keep_cols],
        on="sleep_episode_id",
        how="left",
        validate="one_to_one",
    )
)

design_c_stage1_df["sleep_start_hour"] = (
    design_c_stage1_df["sleep_start_datetime"].dt.hour
    + design_c_stage1_df["sleep_start_datetime"].dt.minute / 60
    + design_c_stage1_df["sleep_start_datetime"].dt.second / 3600
)

design_c_stage1_df["sleep_start_dayofweek"] = design_c_stage1_df[
    "sleep_start_datetime"
].dt.dayofweek

design_c_stage1_df["sleep_start_month"] = design_c_stage1_df[
    "sleep_start_datetime"
].dt.month

design_c_stage1_df["sleep_start_dayofweek_sin"] = np.sin(
    2 * np.pi * design_c_stage1_df["sleep_start_dayofweek"] / 7
)
design_c_stage1_df["sleep_start_dayofweek_cos"] = np.cos(
    2 * np.pi * design_c_stage1_df["sleep_start_dayofweek"] / 7
)
design_c_stage1_df["sleep_start_month_sin"] = np.sin(
    2 * np.pi * design_c_stage1_df["sleep_start_month"] / 12
)
design_c_stage1_df["sleep_start_month_cos"] = np.cos(
    2 * np.pi * design_c_stage1_df["sleep_start_month"] / 12
)

timing_feature_cols = [
    "pre_sleep_window_hours",
    "sleep_start_hour",
    "sleep_start_dayofweek_sin",
    "sleep_start_dayofweek_cos",
    "sleep_start_month_sin",
    "sleep_start_month_cos",
]

intraday_feature_cols = [
    col for col in intraday_features_df.columns
    if col not in [
        "sleep_episode_id",
        ID_COL,
        "sleep_start_datetime",
        "pre_sleep_window_start",
        "pre_sleep_window_end",
        "pre_sleep_window_hours",
    ]
]

stage1_base_feature_cols = (
    intraday_feature_cols
    + timing_feature_cols
    + previous_day_feature_cols
)

missing_indicator_cols = []
for feature in stage1_base_feature_cols:
    missing_col = f"{feature}_missing_ind"
    design_c_stage1_df[missing_col] = design_c_stage1_df[feature].isna().astype(int)
    missing_indicator_cols.append(missing_col)

stage1_feature_cols = stage1_base_feature_cols + missing_indicator_cols

stage1_missing_summary = (
    design_c_stage1_df[stage1_base_feature_cols]
    .isna()
    .mean()
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "feature"})
    .sort_values(["missing_rate", "feature"], ascending=[False, True])
)

stage1_feature_columns_df = pd.DataFrame(
    {
        "feature": stage1_feature_cols,
        "feature_group": [
            (
                "missing_indicator"
                if feature.endswith("_missing_ind")
                else "previous_day_daily"
                if feature.startswith("previous_day_")
                else "timing"
                if feature in timing_feature_cols
                else "pre_sleep_intraday"
            )
            for feature in stage1_feature_cols
        ],
    }
)

stage1_summary = pd.DataFrame(
    [
        {"metric": "rows", "value": len(design_c_stage1_df)},
        {"metric": "participants", "value": design_c_stage1_df[ID_COL].nunique()},
        {"metric": "features_without_missing_indicators", "value": len(stage1_base_feature_cols)},
        {"metric": "missing_indicator_features", "value": len(missing_indicator_cols)},
        {"metric": "total_features", "value": len(stage1_feature_cols)},
        {"metric": "target_mean", "value": design_c_stage1_df[TARGET_COL].mean()},
        {"metric": "target_0", "value": int((design_c_stage1_df[TARGET_COL] == 0).sum())},
        {"metric": "target_1", "value": int((design_c_stage1_df[TARGET_COL] == 1).sum())},
        {
            "metric": "previous_day_daily_join_coverage",
            "value": float(design_c_stage1_df["previous_day_daily_joined"].mean()),
        },
        {
            "metric": "steps_record_coverage",
            "value": float((design_c_stage1_df["steps_pre_sleep_record_count"] > 0).mean()),
        },
        {
            "metric": "calories_record_coverage",
            "value": float((design_c_stage1_df["calories_pre_sleep_record_count"] > 0).mean()),
        },
        {
            "metric": "heart_rate_record_coverage",
            "value": float((design_c_stage1_df["heart_rate_pre_sleep_record_count"] > 0).mean()),
        },
    ]
)

metadata = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset_name": "pre_sleep_design_c_stage1",
    "prediction_target": "good_sleep_label for the sleep episode starting at sleep_start_datetime",
    "prediction_cutoff": "sleep_start_datetime",
    "strict_rule": "Only features available before sleep_start_datetime are included.",
    "rows": int(len(design_c_stage1_df)),
    "participants": int(design_c_stage1_df[ID_COL].nunique()),
    "feature_count": int(len(stage1_feature_cols)),
    "base_feature_count": int(len(stage1_base_feature_cols)),
    "missing_indicator_count": int(len(missing_indicator_cols)),
    "feature_groups": {
        "pre_sleep_intraday": intraday_feature_cols,
        "timing": timing_feature_cols,
        "previous_day_daily": previous_day_feature_cols,
        "missing_indicators": missing_indicator_cols,
    },
    "source_files": {
        "sleep_target": str(SLEEP_TARGET_PATH.relative_to(PROJECT_ROOT)),
        "modeling_daily": str(MODELING_DAILY_PATH.relative_to(PROJECT_ROOT)),
        "design_feature_spec": str(DESIGN_SPEC_PATH.relative_to(PROJECT_ROOT)),
    },
    "mongo_source": {
        "database": MONGO_DB_NAME,
        "collection": MONGO_COLLECTION_NAME,
        "types": INTRADAY_TYPES,
        "index_expected": "{ id: 1, type: 1, data.dateTime: 1 }",
        "aggregation_method": "MongoDB server-side aggregation, validated against raw-fetch sample.",
    },
}

design_c_stage1_df.to_csv(STAGE1_DATASET_PATH, index=False, encoding="utf-8-sig")
stage1_feature_columns_df.to_csv(STAGE1_FEATURE_COLUMNS_PATH, index=False, encoding="utf-8-sig")
stage1_missing_summary.to_csv(STAGE1_MISSING_SUMMARY_PATH, index=False, encoding="utf-8-sig")
STAGE1_METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

log_entry = f"""

## 2026-06-29 - Pre-sleep Design C Stage 1 dataset created

Created strict pre-sleep forecasting Stage 1 dataset.

- Notebook: `notebooks/11_create_pre_sleep_forecasting_dataset.ipynb`
- Dataset: `{STAGE1_DATASET_PATH.relative_to(PROJECT_ROOT)}`
- Feature columns: `{STAGE1_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT)}`
- Missing summary: `{STAGE1_MISSING_SUMMARY_PATH.relative_to(PROJECT_ROOT)}`
- Metadata: `{STAGE1_METADATA_PATH.relative_to(PROJECT_ROOT)}`
- Rows: {len(design_c_stage1_df)}
- Participants: {design_c_stage1_df[ID_COL].nunique()}
- Features: {len(stage1_feature_cols)}
- Base features: {len(stage1_base_feature_cols)}
- Missing indicators: {len(missing_indicator_cols)}
- Target mean: {design_c_stage1_df[TARGET_COL].mean():.4f}
- Previous-day daily join coverage: {design_c_stage1_df["previous_day_daily_joined"].mean():.4f}
- Intraday coverage:
  - steps: {(design_c_stage1_df["steps_pre_sleep_record_count"] > 0).mean():.4f}
  - calories: {(design_c_stage1_df["calories_pre_sleep_record_count"] > 0).mean():.4f}
  - heart_rate: {(design_c_stage1_df["heart_rate_pre_sleep_record_count"] > 0).mean():.4f}
- Leakage rule: all intraday predictors are aggregated with `timestamp < sleep_start_datetime`.
- Server-side MongoDB aggregation was validated against raw-fetch aggregation on sample episodes with zero mismatched features.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("=== Design C Stage 1 Dataset Saved ===")
print("dataset:", STAGE1_DATASET_PATH.relative_to(PROJECT_ROOT), STAGE1_DATASET_PATH.exists())
print("feature columns:", STAGE1_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT), STAGE1_FEATURE_COLUMNS_PATH.exists())
print("missing summary:", STAGE1_MISSING_SUMMARY_PATH.relative_to(PROJECT_ROOT), STAGE1_MISSING_SUMMARY_PATH.exists())
print("metadata:", STAGE1_METADATA_PATH.relative_to(PROJECT_ROOT), STAGE1_METADATA_PATH.exists())

print("\n=== Stage 1 Summary ===")
display(stage1_summary)

print("\n=== Feature Group Counts ===")
display(stage1_feature_columns_df["feature_group"].value_counts().reset_index(name="features"))

print("\n=== Missing Summary ===")
display(stage1_missing_summary)

print("\n=== Dataset Preview ===")
display(design_c_stage1_df.head())

print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Design C Stage 1 Dataset Saved ===
dataset: data\processed\pre_sleep_forecasting\design_c_stage1\pre_sleep_design_c_stage1_dataset.csv True
feature columns: data\processed\pre_sleep_forecasting\design_c_stage1\pre_sleep_design_c_stage1_feature_columns.csv True
missing summary: data\processed\pre_sleep_forecasting\design_c_stage1\pre_sleep_design_c_stage1_missing_summary.csv True
metadata: data\processed\pre_sleep_forecasting\design_c_stage1\metadata.json True

=== Stage 1 Summary ===


,metric,value
0,rows,3551.000000
1,participants,69.000000
2,features_without_missing_indicators,35.000000
3,missing_indicator_features,35.000000
4,total_features,70.000000
5,target_mean,0.393692
6,target_0,2153.000000
7,target_1,1398.000000
8,previous_day_daily_join_coverage,0.873275
9,steps_record_coverage,0.952126



=== Feature Group Counts ===


,feature_group,features
0,missing_indicator,35
1,pre_sleep_intraday,18
2,previous_day_daily,11
3,timing,6



=== Missing Summary ===


,feature,missing_rate
32,previous_day_steps_record_count,0.128978
31,previous_day_steps_sum,0.128978
34,previous_day_calories_record_count,0.126725
33,previous_day_calories_sum,0.126725
27,previous_day_lightly_active_minutes_sum,0.126725
28,previous_day_moderately_active_minutes_sum,0.126725
25,previous_day_resting_hr_error_mean,0.126725
26,previous_day_resting_hr_record_count,0.126725
24,previous_day_resting_hr_resting_heart_rate_mean,0.126725
29,previous_day_sedentary_minutes_sum,0.126725



=== Dataset Preview ===


,sleep_episode_id,participant_object_id,calendar_date,sleep_start_datetime,sleep_end_datetime,sleep_start_date,sleep_end_date,prediction_cutoff_datetime,good_sleep_label,cross_midnight,...,previous_day_resting_hr_error_mean_missing_ind,previous_day_resting_hr_record_count_missing_ind,previous_day_lightly_active_minutes_sum_missing_ind,previous_day_moderately_active_minutes_sum_missing_ind,previous_day_sedentary_minutes_sum_missing_ind,previous_day_very_active_minutes_sum_missing_ind,previous_day_steps_sum_missing_ind,previous_day_steps_record_count_missing_ind,previous_day_calories_sum_missing_ind,previous_day_calories_record_count_missing_ind
0,621e2e8e67b776a24055b564__20210524004000__2021...,621e2e8e67b776a24055b564,2021-05-24,2021-05-24 00:40:00,2021-05-24 09:21:00,2021-05-24,2021-05-24,2021-05-24 00:40:00,1,False,...,1,1,1,1,1,1,1,1,1,1
1,621e2e8e67b776a24055b564__20210524234830__2021...,621e2e8e67b776a24055b564,2021-05-25,2021-05-24 23:48:30,2021-05-25 08:56:30,2021-05-24,2021-05-25,2021-05-24 23:48:30,1,True,...,1,1,1,1,1,1,1,1,1,1
2,621e2e8e67b776a24055b564__20210525234630__2021...,621e2e8e67b776a24055b564,2021-05-26,2021-05-25 23:46:30,2021-05-26 09:06:30,2021-05-25,2021-05-26,2021-05-25 23:46:30,1,True,...,0,0,0,0,0,0,0,0,0,0
3,621e2e8e67b776a24055b564__20210526232130__2021...,621e2e8e67b776a24055b564,2021-05-27,2021-05-26 23:21:30,2021-05-27 09:48:30,2021-05-26,2021-05-27,2021-05-26 23:21:30,1,True,...,0,0,0,0,0,0,0,0,0,0
4,621e2e8e67b776a24055b564__20210527233700__2021...,621e2e8e67b776a24055b564,2021-05-28,2021-05-27 23:37:00,2021-05-28 08:58:30,2021-05-27,2021-05-28,2021-05-27 23:37:00,1,True,...,0,0,0,0,0,0,0,0,0,0



log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True


In [11]:
# Cell 7. Create participant-level train/validation/test split for Design C Stage 1
# 원하는 결과:
# - participant 단위로 train/validation/test split을 새로 만든다.
# - 같은 participant가 여러 split에 섞이지 않는지 확인한다.
# - split별 row 수, participant 수, target distribution을 확인한다.
# - 아직 파일 저장은 하지 않는다.

from sklearn.model_selection import train_test_split

SPLIT_RANDOM_STATE = 2026
TEST_SIZE = 0.20
VALIDATION_SIZE_WITHIN_TRAIN_VALIDATION = 0.15

split_work_df = design_c_stage1_df.copy()

participant_target_summary = (
    split_work_df
    .groupby(ID_COL)
    .agg(
        rows=(TARGET_COL, "size"),
        target_mean=(TARGET_COL, "mean"),
        target_0=(TARGET_COL, lambda x: int((x == 0).sum())),
        target_1=(TARGET_COL, lambda x: int((x == 1).sum())),
        min_sleep_start=("sleep_start_datetime", "min"),
        max_sleep_start=("sleep_start_datetime", "max"),
    )
    .reset_index()
)

participant_target_summary["participant_target_bin"] = pd.qcut(
    participant_target_summary["target_mean"].rank(method="first"),
    q=min(5, len(participant_target_summary)),
    labels=False,
    duplicates="drop",
)

train_validation_participants, test_participants = train_test_split(
    participant_target_summary[ID_COL],
    test_size=TEST_SIZE,
    random_state=SPLIT_RANDOM_STATE,
    stratify=participant_target_summary["participant_target_bin"],
)

train_validation_summary = participant_target_summary[
    participant_target_summary[ID_COL].isin(train_validation_participants)
].copy()

train_participants, validation_participants = train_test_split(
    train_validation_summary[ID_COL],
    test_size=VALIDATION_SIZE_WITHIN_TRAIN_VALIDATION,
    random_state=SPLIT_RANDOM_STATE,
    stratify=train_validation_summary["participant_target_bin"],
)

split_map = {}
split_map.update({participant_id: "train" for participant_id in train_participants})
split_map.update({participant_id: "validation" for participant_id in validation_participants})
split_map.update({participant_id: "test" for participant_id in test_participants})

split_work_df["pre_sleep_split"] = split_work_df[ID_COL].map(split_map)

split_summary = (
    split_work_df
    .groupby("pre_sleep_split")
    .agg(
        rows=(TARGET_COL, "size"),
        participants=(ID_COL, "nunique"),
        target_mean=(TARGET_COL, "mean"),
        target_0=(TARGET_COL, lambda x: int((x == 0).sum())),
        target_1=(TARGET_COL, lambda x: int((x == 1).sum())),
        min_sleep_start=("sleep_start_datetime", "min"),
        max_sleep_start=("sleep_start_datetime", "max"),
    )
    .reset_index()
    .sort_values("pre_sleep_split")
)

split_overlap_rows = []
split_names = sorted(split_work_df["pre_sleep_split"].dropna().unique())

for i, split_a in enumerate(split_names):
    participants_a = set(
        split_work_df.loc[split_work_df["pre_sleep_split"] == split_a, ID_COL]
    )
    for split_b in split_names[i + 1:]:
        participants_b = set(
            split_work_df.loc[split_work_df["pre_sleep_split"] == split_b, ID_COL]
        )
        overlap = sorted(participants_a & participants_b)
        split_overlap_rows.append(
            {
                "split_pair": f"{split_a} vs {split_b}",
                "overlap_participants": len(overlap),
                "example_participants": overlap[:5],
            }
        )

split_overlap_df = pd.DataFrame(split_overlap_rows)

missing_split_rows = split_work_df["pre_sleep_split"].isna().sum()

participant_split_summary = (
    split_work_df[[ID_COL, "pre_sleep_split"]]
    .drop_duplicates()
    .groupby("pre_sleep_split")
    .size()
    .reset_index(name="participants")
)

print("=== Participant-Level Split Summary ===")
print("random_state:", SPLIT_RANDOM_STATE)
print("test_size:", TEST_SIZE)
print("validation_size_within_train_validation:", VALIDATION_SIZE_WITHIN_TRAIN_VALIDATION)
display(split_summary)

print("\n=== Participant Split Counts ===")
display(participant_split_summary)

print("\n=== Participant Split Overlap Check ===")
display(split_overlap_df)

print("\n=== Participant Target Summary Preview ===")
display(participant_target_summary.head(20))

print("\n=== Validation Checks ===")
print("missing split rows:", missing_split_rows)
print("total rows:", len(split_work_df))
print("unique participants:", split_work_df[ID_COL].nunique())
print("split participants:", sum(split_summary["participants"]))
print("overlap participant pairs:", int(split_overlap_df["overlap_participants"].sum()))

if (
    missing_split_rows == 0
    and split_overlap_df["overlap_participants"].sum() == 0
    and sum(split_summary["participants"]) == split_work_df[ID_COL].nunique()
):
    print("participant-level split validation passed")
else:
    print("split validation failed; inspect before tensor creation")

=== Participant-Level Split Summary ===
random_state: 2026
test_size: 0.2
validation_size_within_train_validation: 0.15


,pre_sleep_split,rows,participants,target_mean,target_0,target_1,min_sleep_start,max_sleep_start
0,test,881,14,0.360953,563,318,2021-05-24 00:26:30,2022-01-21 02:25:00
1,train,2323,46,0.412398,1365,958,2021-05-24 00:08:30,2022-01-21 23:04:00
2,validation,347,9,0.351585,225,122,2021-05-24 00:16:30,2022-01-19 03:07:00



=== Participant Split Counts ===


,pre_sleep_split,participants
0,test,14
1,train,46
2,validation,9



=== Participant Split Overlap Check ===


,split_pair,overlap_participants,example_participants
0,test vs train,0,[]
1,test vs validation,0,[]
2,train vs validation,0,[]



=== Participant Target Summary Preview ===


,participant_object_id,rows,target_mean,target_0,target_1,min_sleep_start,max_sleep_start,participant_target_bin
0,621e2e8e67b776a24055b564,63,0.888889,7,56,2021-05-24 00:40:00,2021-08-01 00:10:00,4
1,621e2eaf67b776a2406b14ac,83,0.518072,40,43,2021-10-28 22:15:00,2022-01-21 23:04:00,3
2,621e2ed667b776a24085d8d1,76,0.526316,36,40,2021-05-24 22:32:30,2021-08-16 21:53:00,3
3,621e2ef567b776a24099f889,1,1.000000,0,1,2021-10-18 22:41:30,2021-10-18 22:41:30,4
4,621e2efa67b776a2409dd1c3,66,0.469697,35,31,2021-05-24 23:03:00,2021-09-28 22:27:30,3
5,621e2f1b67b776a240b3d87c,76,0.565789,33,43,2021-11-07 00:31:00,2022-01-20 23:11:00,3
6,621e2f3967b776a240c654db,62,0.612903,24,38,2021-05-25 00:03:00,2021-08-01 09:47:30,4
7,621e2f5767b776a240d8f9d6,22,0.500000,11,11,2021-11-15 01:42:00,2021-12-07 02:16:00,3
8,621e2f6167b776a240e082a9,45,0.288889,32,13,2021-05-26 01:34:00,2021-07-29 01:22:00,2
9,621e2f7a67b776a240f14425,53,0.320755,36,17,2021-05-24 00:08:30,2021-07-28 23:05:00,2



=== Validation Checks ===
missing split rows: 0
total rows: 3551
unique participants: 69
split participants: 69
overlap participant pairs: 0
participant-level split validation passed


In [12]:
# Cell 8. Save split dataset and create scaled tabular tensors for MLP
# 원하는 결과:
# - pre_sleep_split이 포함된 Stage 1 dataset을 저장한다.
# - train 기준 median imputation + StandardScaler를 적용한다.
# - MLP용 train/validation/test npz tensor를 저장한다.
# - NaN/Inf, split별 target distribution, feature count를 확인한다.
# - log/YYYY-MM-DD.md를 업데이트한다.

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

SPLIT_DATASET_PATH = DESIGN_C_OUTPUT_DIR / "pre_sleep_design_c_stage1_dataset_with_split.csv"
SCALER_PATH = DESIGN_C_OUTPUT_DIR / "pre_sleep_design_c_stage1_standard_scaler.joblib"
IMPUTER_PATH = DESIGN_C_OUTPUT_DIR / "pre_sleep_design_c_stage1_median_imputer.joblib"
MLP_TENSOR_DIR = DESIGN_C_OUTPUT_DIR / "mlp_current_day"
TENSOR_SUMMARY_PATH = DESIGN_C_OUTPUT_DIR / "pre_sleep_design_c_stage1_tensor_summary.csv"

MLP_TENSOR_DIR.mkdir(parents=True, exist_ok=True)

design_c_stage1_split_df = split_work_df.copy()

# Reattach all saved Stage 1 feature engineering columns if split_work_df was made before final save.
design_c_stage1_split_df = design_c_stage1_df.merge(
    split_work_df[["sleep_episode_id", "pre_sleep_split"]],
    on="sleep_episode_id",
    how="left",
    validate="one_to_one",
)

train_mask = design_c_stage1_split_df["pre_sleep_split"] == "train"

X_raw = design_c_stage1_split_df[stage1_feature_cols].copy()
y = design_c_stage1_split_df[TARGET_COL].astype(np.int64).to_numpy()

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_imputed = imputer.fit_transform(X_raw.loc[train_mask])
scaler.fit(X_train_imputed)

X_imputed = imputer.transform(X_raw)
X_scaled = scaler.transform(X_imputed).astype(np.float32)

scaled_check = pd.DataFrame(
    [
        {"metric": "rows", "value": X_scaled.shape[0]},
        {"metric": "features", "value": X_scaled.shape[1]},
        {"metric": "nan_count", "value": int(np.isnan(X_scaled).sum())},
        {"metric": "inf_count", "value": int(np.isinf(X_scaled).sum())},
        {
            "metric": "max_abs_train_scaled_mean",
            "value": float(np.abs(X_scaled[train_mask.to_numpy()].mean(axis=0)).max()),
        },
        {
            "metric": "min_train_scaled_std",
            "value": float(X_scaled[train_mask.to_numpy()].std(axis=0).min()),
        },
        {
            "metric": "max_train_scaled_std",
            "value": float(X_scaled[train_mask.to_numpy()].std(axis=0).max()),
        },
    ]
)

tensor_summary_rows = []

for split_name in ["train", "validation", "test"]:
    split_mask = design_c_stage1_split_df["pre_sleep_split"].to_numpy() == split_name

    X_split = X_scaled[split_mask]
    y_split = y[split_mask]
    episode_ids_split = design_c_stage1_split_df.loc[split_mask, "sleep_episode_id"].astype(str).to_numpy()
    participant_ids_split = design_c_stage1_split_df.loc[split_mask, ID_COL].astype(str).to_numpy()

    output_path = MLP_TENSOR_DIR / f"{split_name}.npz"

    np.savez_compressed(
        output_path,
        X=X_split,
        y=y_split,
        sleep_episode_id=episode_ids_split,
        participant_object_id=participant_ids_split,
        feature_columns=np.array(stage1_feature_cols, dtype=object),
    )

    tensor_summary_rows.append(
        {
            "tensor_type": "mlp_current_day",
            "split": split_name,
            "samples": int(X_split.shape[0]),
            "features": int(X_split.shape[1]),
            "participants": int(pd.Series(participant_ids_split).nunique()),
            "target_0": int((y_split == 0).sum()),
            "target_1": int((y_split == 1).sum()),
            "target_mean": float(y_split.mean()),
            "nan_count": int(np.isnan(X_split).sum()),
            "inf_count": int(np.isinf(X_split).sum()),
            "path": str(output_path.relative_to(PROJECT_ROOT)),
        }
    )

tensor_summary_df = pd.DataFrame(tensor_summary_rows)

design_c_stage1_split_df.to_csv(SPLIT_DATASET_PATH, index=False, encoding="utf-8-sig")
joblib.dump(imputer, IMPUTER_PATH)
joblib.dump(scaler, SCALER_PATH)
tensor_summary_df.to_csv(TENSOR_SUMMARY_PATH, index=False, encoding="utf-8-sig")

metadata["split"] = {
    "split_column": "pre_sleep_split",
    "split_random_state": SPLIT_RANDOM_STATE,
    "test_size": TEST_SIZE,
    "validation_size_within_train_validation": VALIDATION_SIZE_WITHIN_TRAIN_VALIDATION,
    "split_strategy": "participant-level split stratified by participant target-rate bins",
}
metadata["preprocessing"] = {
    "imputer": "SimpleImputer(strategy='median'), fit on train only",
    "scaler": "StandardScaler, fit on train only after imputation",
}
metadata["tensor_outputs"] = {
    "mlp_current_day": str(MLP_TENSOR_DIR.relative_to(PROJECT_ROOT)),
    "tensor_summary": str(TENSOR_SUMMARY_PATH.relative_to(PROJECT_ROOT)),
}
STAGE1_METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

log_entry = f"""

## 2026-06-29 - Pre-sleep Design C Stage 1 split and MLP tensors

Created participant-level split and tabular MLP tensors for strict pre-sleep Stage 1 dataset.

- Split dataset: `{SPLIT_DATASET_PATH.relative_to(PROJECT_ROOT)}`
- Tensor dir: `{MLP_TENSOR_DIR.relative_to(PROJECT_ROOT)}`
- Tensor summary: `{TENSOR_SUMMARY_PATH.relative_to(PROJECT_ROOT)}`
- Imputer: `{IMPUTER_PATH.relative_to(PROJECT_ROOT)}`
- Scaler: `{SCALER_PATH.relative_to(PROJECT_ROOT)}`
- Split strategy: participant-level split, random_state={SPLIT_RANDOM_STATE}
- Train/validation/test samples: {tensor_summary_df.set_index("split").loc["train", "samples"]} / {tensor_summary_df.set_index("split").loc["validation", "samples"]} / {tensor_summary_df.set_index("split").loc["test", "samples"]}
- Feature count: {len(stage1_feature_cols)}
- NaN/Inf after preprocessing: {int(np.isnan(X_scaled).sum())} / {int(np.isinf(X_scaled).sum())}
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("=== Stage 1 Split Dataset And MLP Tensors Saved ===")
print("split dataset:", SPLIT_DATASET_PATH.relative_to(PROJECT_ROOT), SPLIT_DATASET_PATH.exists())
print("feature columns:", STAGE1_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT), STAGE1_FEATURE_COLUMNS_PATH.exists())
print("imputer:", IMPUTER_PATH.relative_to(PROJECT_ROOT), IMPUTER_PATH.exists())
print("scaler:", SCALER_PATH.relative_to(PROJECT_ROOT), SCALER_PATH.exists())
print("tensor summary:", TENSOR_SUMMARY_PATH.relative_to(PROJECT_ROOT), TENSOR_SUMMARY_PATH.exists())

print("\n=== Saved Tensor Files ===")
for split_name in ["train", "validation", "test"]:
    path = MLP_TENSOR_DIR / f"{split_name}.npz"
    print(path.relative_to(PROJECT_ROOT), path.exists())

print("\n=== Scaled Feature Check ===")
display(scaled_check)

print("\n=== Tensor Summary ===")
display(tensor_summary_df)

print("\n=== Split Summary Recheck ===")
display(split_summary)

print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Stage 1 Split Dataset And MLP Tensors Saved ===
split dataset: data\processed\pre_sleep_forecasting\design_c_stage1\pre_sleep_design_c_stage1_dataset_with_split.csv True
feature columns: data\processed\pre_sleep_forecasting\design_c_stage1\pre_sleep_design_c_stage1_feature_columns.csv True
imputer: data\processed\pre_sleep_forecasting\design_c_stage1\pre_sleep_design_c_stage1_median_imputer.joblib True
scaler: data\processed\pre_sleep_forecasting\design_c_stage1\pre_sleep_design_c_stage1_standard_scaler.joblib True
tensor summary: data\processed\pre_sleep_forecasting\design_c_stage1\pre_sleep_design_c_stage1_tensor_summary.csv True

=== Saved Tensor Files ===
data\processed\pre_sleep_forecasting\design_c_stage1\mlp_current_day\train.npz True
data\processed\pre_sleep_forecasting\design_c_stage1\mlp_current_day\validation.npz True
data\processed\pre_sleep_forecasting\design_c_stage1\mlp_current_day\test.npz True

=== Scaled Feature Check ===


,metric,value
0,rows,3.551000e+03
1,features,7.000000e+01
2,nan_count,0.000000e+00
3,inf_count,0.000000e+00
4,max_abs_train_scaled_mean,4.684212e-07
5,min_train_scaled_std,0.000000e+00
6,max_train_scaled_std,1.000011e+00



=== Tensor Summary ===


,tensor_type,split,samples,features,participants,target_0,target_1,target_mean,nan_count,inf_count,path
0,mlp_current_day,train,2323,70,46,1365,958,0.412398,0,0,data\processed\pre_sleep_forecasting\design_c_...
1,mlp_current_day,validation,347,70,9,225,122,0.351585,0,0,data\processed\pre_sleep_forecasting\design_c_...
2,mlp_current_day,test,881,70,14,563,318,0.360953,0,0,data\processed\pre_sleep_forecasting\design_c_...



=== Split Summary Recheck ===


,pre_sleep_split,rows,participants,target_mean,target_0,target_1,min_sleep_start,max_sleep_start
0,test,881,14,0.360953,563,318,2021-05-24 00:26:30,2022-01-21 02:25:00
1,train,2323,46,0.412398,1365,958,2021-05-24 00:08:30,2022-01-21 23:04:00
2,validation,347,9,0.351585,225,122,2021-05-24 00:16:30,2022-01-19 03:07:00



log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True


In [13]:
# Cell 9. Remove train zero-variance features and resave final Stage 1 MLP tensors
# 원하는 결과:
# - train split 기준 zero-variance feature를 제거한다.
# - 최종 feature list와 제거 feature list를 저장한다.
# - 최종 MLP tensor를 다시 저장한다.
# - NaN/Inf와 scaling 상태를 재확인한다.
# - log/YYYY-MM-DD.md를 업데이트한다.

ZERO_VARIANCE_FEATURES_PATH = DESIGN_C_OUTPUT_DIR / "pre_sleep_design_c_stage1_zero_variance_removed_features.csv"
FINAL_FEATURE_COLUMNS_PATH = DESIGN_C_OUTPUT_DIR / "pre_sleep_design_c_stage1_final_feature_columns.csv"
FINAL_MLP_TENSOR_DIR = DESIGN_C_OUTPUT_DIR / "mlp_current_day_final"
FINAL_TENSOR_SUMMARY_PATH = DESIGN_C_OUTPUT_DIR / "pre_sleep_design_c_stage1_final_tensor_summary.csv"

FINAL_MLP_TENSOR_DIR.mkdir(parents=True, exist_ok=True)

train_scaled = X_scaled[train_mask.to_numpy()]
train_feature_std = train_scaled.std(axis=0)

zero_variance_mask = train_feature_std <= 1e-8
zero_variance_features = [
    feature for feature, is_zero in zip(stage1_feature_cols, zero_variance_mask)
    if is_zero
]

final_stage1_feature_cols = [
    feature for feature, is_zero in zip(stage1_feature_cols, zero_variance_mask)
    if not is_zero
]

X_scaled_final = X_scaled[:, ~zero_variance_mask].astype(np.float32)

zero_variance_features_df = pd.DataFrame(
    {
        "removed_feature": zero_variance_features,
        "train_scaled_std": train_feature_std[zero_variance_mask],
    }
)

final_feature_columns_df = stage1_feature_columns_df[
    stage1_feature_columns_df["feature"].isin(final_stage1_feature_cols)
].copy()

final_scaled_check = pd.DataFrame(
    [
        {"metric": "input_features", "value": len(stage1_feature_cols)},
        {"metric": "zero_variance_features", "value": len(zero_variance_features)},
        {"metric": "final_features", "value": len(final_stage1_feature_cols)},
        {"metric": "rows", "value": X_scaled_final.shape[0]},
        {"metric": "nan_count", "value": int(np.isnan(X_scaled_final).sum())},
        {"metric": "inf_count", "value": int(np.isinf(X_scaled_final).sum())},
        {
            "metric": "max_abs_train_scaled_mean",
            "value": float(np.abs(X_scaled_final[train_mask.to_numpy()].mean(axis=0)).max()),
        },
        {
            "metric": "min_train_scaled_std",
            "value": float(X_scaled_final[train_mask.to_numpy()].std(axis=0).min()),
        },
        {
            "metric": "max_train_scaled_std",
            "value": float(X_scaled_final[train_mask.to_numpy()].std(axis=0).max()),
        },
    ]
)

final_tensor_summary_rows = []

for split_name in ["train", "validation", "test"]:
    split_mask = design_c_stage1_split_df["pre_sleep_split"].to_numpy() == split_name

    X_split = X_scaled_final[split_mask]
    y_split = y[split_mask]
    episode_ids_split = design_c_stage1_split_df.loc[split_mask, "sleep_episode_id"].astype(str).to_numpy()
    participant_ids_split = design_c_stage1_split_df.loc[split_mask, ID_COL].astype(str).to_numpy()

    output_path = FINAL_MLP_TENSOR_DIR / f"{split_name}.npz"

    np.savez_compressed(
        output_path,
        X=X_split,
        y=y_split,
        sleep_episode_id=episode_ids_split,
        participant_object_id=participant_ids_split,
        feature_columns=np.array(final_stage1_feature_cols, dtype=object),
    )

    final_tensor_summary_rows.append(
        {
            "tensor_type": "mlp_current_day_final",
            "split": split_name,
            "samples": int(X_split.shape[0]),
            "features": int(X_split.shape[1]),
            "participants": int(pd.Series(participant_ids_split).nunique()),
            "target_0": int((y_split == 0).sum()),
            "target_1": int((y_split == 1).sum()),
            "target_mean": float(y_split.mean()),
            "nan_count": int(np.isnan(X_split).sum()),
            "inf_count": int(np.isinf(X_split).sum()),
            "path": str(output_path.relative_to(PROJECT_ROOT)),
        }
    )

final_tensor_summary_df = pd.DataFrame(final_tensor_summary_rows)

zero_variance_features_df.to_csv(ZERO_VARIANCE_FEATURES_PATH, index=False, encoding="utf-8-sig")
final_feature_columns_df.to_csv(FINAL_FEATURE_COLUMNS_PATH, index=False, encoding="utf-8-sig")
final_tensor_summary_df.to_csv(FINAL_TENSOR_SUMMARY_PATH, index=False, encoding="utf-8-sig")

metadata["zero_variance_removal"] = {
    "criterion": "train split scaled std <= 1e-8",
    "removed_feature_count": int(len(zero_variance_features)),
    "final_feature_count": int(len(final_stage1_feature_cols)),
    "removed_features_path": str(ZERO_VARIANCE_FEATURES_PATH.relative_to(PROJECT_ROOT)),
    "final_feature_columns_path": str(FINAL_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT)),
}
metadata["tensor_outputs"]["mlp_current_day_final"] = str(FINAL_MLP_TENSOR_DIR.relative_to(PROJECT_ROOT))
metadata["tensor_outputs"]["final_tensor_summary"] = str(FINAL_TENSOR_SUMMARY_PATH.relative_to(PROJECT_ROOT))

STAGE1_METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

log_entry = f"""

## 2026-06-29 - Pre-sleep Design C Stage 1 final MLP tensors

Removed train zero-variance features and saved final MLP tensors.

- Input features: {len(stage1_feature_cols)}
- Removed zero-variance features: {len(zero_variance_features)}
- Final features: {len(final_stage1_feature_cols)}
- Removed feature list: `{ZERO_VARIANCE_FEATURES_PATH.relative_to(PROJECT_ROOT)}`
- Final feature columns: `{FINAL_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT)}`
- Final tensor dir: `{FINAL_MLP_TENSOR_DIR.relative_to(PROJECT_ROOT)}`
- Final tensor summary: `{FINAL_TENSOR_SUMMARY_PATH.relative_to(PROJECT_ROOT)}`
- NaN/Inf after final tensor creation: {int(np.isnan(X_scaled_final).sum())} / {int(np.isinf(X_scaled_final).sum())}
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("=== Zero-Variance Feature Removal Complete ===")
print("zero-variance removed features:", ZERO_VARIANCE_FEATURES_PATH.relative_to(PROJECT_ROOT), ZERO_VARIANCE_FEATURES_PATH.exists())
print("final feature columns:", FINAL_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT), FINAL_FEATURE_COLUMNS_PATH.exists())
print("final tensor summary:", FINAL_TENSOR_SUMMARY_PATH.relative_to(PROJECT_ROOT), FINAL_TENSOR_SUMMARY_PATH.exists())

print("\n=== Removed Zero-Variance Features ===")
display(zero_variance_features_df)

print("\n=== Final Scaled Feature Check ===")
display(final_scaled_check)

print("\n=== Final Tensor Summary ===")
display(final_tensor_summary_df)

print("\n=== Saved Final Tensor Files ===")
for split_name in ["train", "validation", "test"]:
    path = FINAL_MLP_TENSOR_DIR / f"{split_name}.npz"
    print(path.relative_to(PROJECT_ROOT), path.exists())

print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Zero-Variance Feature Removal Complete ===
zero-variance removed features: data\processed\pre_sleep_forecasting\design_c_stage1\pre_sleep_design_c_stage1_zero_variance_removed_features.csv True
final feature columns: data\processed\pre_sleep_forecasting\design_c_stage1\pre_sleep_design_c_stage1_final_feature_columns.csv True
final tensor summary: data\processed\pre_sleep_forecasting\design_c_stage1\pre_sleep_design_c_stage1_final_tensor_summary.csv True

=== Removed Zero-Variance Features ===


,removed_feature,train_scaled_std
0,previous_day_resting_hr_record_count,0.0
1,previous_day_calories_record_count,0.0
2,steps_pre_sleep_record_count_missing_ind,0.0
3,steps_pre_sleep_active_record_count_missing_ind,0.0
4,calories_pre_sleep_record_count_missing_ind,0.0
5,heart_rate_pre_sleep_record_count_missing_ind,0.0
6,pre_sleep_window_hours_missing_ind,0.0
7,sleep_start_hour_missing_ind,0.0
8,sleep_start_dayofweek_sin_missing_ind,0.0
9,sleep_start_dayofweek_cos_missing_ind,0.0



=== Final Scaled Feature Check ===


,metric,value
0,input_features,7.000000e+01
1,zero_variance_features,1.200000e+01
2,final_features,5.800000e+01
3,rows,3.551000e+03
4,nan_count,0.000000e+00
5,inf_count,0.000000e+00
6,max_abs_train_scaled_mean,4.684212e-07
7,min_train_scaled_std,9.999912e-01
8,max_train_scaled_std,1.000011e+00



=== Final Tensor Summary ===


,tensor_type,split,samples,features,participants,target_0,target_1,target_mean,nan_count,inf_count,path
0,mlp_current_day_final,train,2323,58,46,1365,958,0.412398,0,0,data\processed\pre_sleep_forecasting\design_c_...
1,mlp_current_day_final,validation,347,58,9,225,122,0.351585,0,0,data\processed\pre_sleep_forecasting\design_c_...
2,mlp_current_day_final,test,881,58,14,563,318,0.360953,0,0,data\processed\pre_sleep_forecasting\design_c_...



=== Saved Final Tensor Files ===
data\processed\pre_sleep_forecasting\design_c_stage1\mlp_current_day_final\train.npz True
data\processed\pre_sleep_forecasting\design_c_stage1\mlp_current_day_final\validation.npz True
data\processed\pre_sleep_forecasting\design_c_stage1\mlp_current_day_final\test.npz True

log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True
